<div style="direction: rtl; white-space: normal; line-height: 1;">

# Notebook 04 — Book Ingestion

### Start Gate — تأیید ورودی‌های پذیرفته‌شده پیش از Ingestion

این Notebook مرحله `Book Ingestion` در نقشه ماژولار Phase 2 است.

Notebook قبلی:

`03_book_inventory.ipynb`

با وضعیت زیر بسته شده است:

`NOTEBOOK 03 FINAL ACCEPTANCE: PASS`

و:

`RESTART KERNEL → RUN ALL GATE: PASS`

در Notebook 03 تعداد `7` فایل فیزیکی در قالب `4` Work مستقل شناسایی شد و Source Selection رسمی انجام گرفت.

ورودی اصلی این Notebook:

`data/phase2/manifests/book_manifest_v1.json`

است.

طبق Inventory پذیرفته‌شده:

- فایل `[3]` → `ACCEPTED_FOR_INGESTION`
- فایل `[4]` → `ACCEPTED_FOR_INGESTION`
- فایل `[6]` → `ACCEPTED_FOR_INGESTION`

سایر فایل‌ها در این Notebook وارد Corpus اصلی نمی‌شوند:

- `[1]` → `AUTHORITATIVE_REFERENCE`
- `[2]` → `QUARANTINED_PENDING_TRANSLATION_QA`
- `[5]` و `[7]` → `NON_CANONICAL`

بنابراین در این Notebook هیچ تصمیم Inventory جدیدی گرفته نمی‌شود و تصمیم‌های Notebook 03 به‌عنوان Source of Truth حفظ می‌شوند.

هدف این Cell فقط اجرای `Start Gate` است:

- شناسایی Root پروژه بدون Hard-code
- خواندن Book Manifest از Disk
- تأیید تعداد Work، Edition و Physical File
- تأیید تصمیم‌های Ingestion
- بررسی وجود واقعی سه فایل پذیرفته‌شده
- تطبیق `SHA-256` فایل‌های پذیرفته‌شده با Manifest

این مرحله هنوز هیچ متن کتابی استخراج یا ذخیره نمی‌کند.

همچنین در این Notebook **Chunking انجام نخواهد شد**. Chunking طبق Roadmap در:

`11_chunking_benchmark.ipynb`

بررسی و انتخاب می‌شود.

**معیار PASS:**

- `book_manifest_v1.json` موجود و معتبر باشد.
- دقیقاً فایل‌های `[3]`، `[4]` و `[6]` برای Ingestion پذیرفته شده باشند.
- هیچ فایل Quarantined یا Non-canonical وارد Candidateها نشود.
- سه فایل پذیرفته‌شده روی Disk موجود باشند.
- Hash واقعی هر سه فایل با Manifest یکسان باشد.

</div>

In [1]:
# Code Cell 01

from pathlib import Path
import hashlib
import json

current_path = Path.cwd().resolve()
search_paths = [current_path, *current_path.parents]

project_root = next(
    (
        path
        for path in search_paths
        if (path / "data" / "phase2" / "manifests").is_dir()
        and (path / "notebooks" / "phase2").is_dir()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Project root could not be detected from the current working directory."
    )

manifest_path = (
    project_root
    / "data"
    / "phase2"
    / "manifests"
    / "book_manifest_v1.json"
)

if not manifest_path.exists():
    raise FileNotFoundError(
        f"Book manifest not found: {manifest_path.relative_to(project_root)}"
    )

with manifest_path.open("r", encoding="utf-8") as file:
    book_manifest = json.load(file)

works = book_manifest.get("works", [])
editions = book_manifest.get("editions", [])
physical_files = book_manifest.get("physical_files", [])

accepted_editions = [
    edition
    for edition in editions
    if edition.get("ingestion_status") == "ACCEPTED_FOR_INGESTION"
]

accepted_indices = sorted(
    edition["file_index"]
    for edition in accepted_editions
)

file_record_by_index = {
    record["file_index"]: record
    for record in physical_files
}

missing_files = []
hash_mismatches = []

for file_index in accepted_indices:
    record = file_record_by_index[file_index]
    path = project_root / record["relative_path"]

    if not path.exists():
        missing_files.append(record["relative_path"])
        continue

    actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()

    if actual_hash != record["file_sha256"]:
        hash_mismatches.append(record["relative_path"])

expected_decisions = {
    1: "AUTHORITATIVE_REFERENCE",
    2: "QUARANTINED_PENDING_TRANSLATION_QA",
    3: "ACCEPTED_FOR_INGESTION",
    4: "ACCEPTED_FOR_INGESTION",
    5: "NON_CANONICAL",
    6: "ACCEPTED_FOR_INGESTION",
    7: "NON_CANONICAL",
}

actual_decisions = {
    edition["file_index"]: edition["ingestion_status"]
    for edition in editions
}

decision_mismatches = {
    index: {
        "expected": expected,
        "actual": actual_decisions.get(index),
    }
    for index, expected in expected_decisions.items()
    if actual_decisions.get(index) != expected
}

start_gate_pass = (
    book_manifest.get("manifest_type") == "phase2_book_inventory"
    and book_manifest.get("manifest_version") == "v1"
    and len(works) == 4
    and len(editions) == 7
    and len(physical_files) == 7
    and accepted_indices == [3, 4, 6]
    and not missing_files
    and not hash_mismatches
    and not decision_mismatches
)

print("NOTEBOOK 04 — BOOK INGESTION START GATE")
print("-" * 80)
print("Project root:", project_root)
print("Manifest:", manifest_path.relative_to(project_root))
print("Works:", len(works))
print("Editions:", len(editions))
print("Physical files:", len(physical_files))
print("Accepted file indices:", accepted_indices)
print("Missing accepted files:", len(missing_files))
print("Accepted file hash mismatches:", len(hash_mismatches))
print("Ingestion decision mismatches:", len(decision_mismatches))

print("\nACCEPTED INGESTION SOURCES")
print("-" * 80)

for edition in accepted_editions:
    file_index = edition["file_index"]
    record = file_record_by_index[file_index]

    print(f"[{file_index}] {record['filename']}")
    print("    Work ID:", edition["work_id"])
    print("    Edition ID:", edition["edition_id"])
    print("    Language:", edition["language"])
    print("    Status:", edition["ingestion_status"])

print()
print(
    "BOOK INGESTION START GATE:",
    "PASS" if start_gate_pass else "FAIL",
)

NOTEBOOK 04 — BOOK INGESTION START GATE
--------------------------------------------------------------------------------
Project root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Manifest: data/phase2/manifests/book_manifest_v1.json
Works: 4
Editions: 7
Physical files: 7
Accepted file indices: [3, 4, 6]
Missing accepted files: 0
Accepted file hash mismatches: 0
Ingestion decision mismatches: 0

ACCEPTED INGESTION SOURCES
--------------------------------------------------------------------------------
[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
    Work ID: book:adult-financial-literacy-course
    Edition ID: book:adult-financial-literacy-course:fa-source
    Language: fa
    Status: ACCEPTED_FOR_INGESTION
[4] سواد مالی برای همه.docx
    Work ID: book:financial-literacy-for-all
    Edition ID: book:financial-literacy-for-all:fa-source
    Language: fa
    Status: ACCEPTED_FOR_INGESTION
[6] نوشتار آموزشی تربیت مربی سواد مالی.docx
    Work ID: boo

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بازاستخراج مستقل منابع و Validation پیش از ساخت Artifact

`BOOK INGESTION START GATE: PASS`

قبل از تولید هر Artifact مربوط به Book Ingestion، سه فایل پذیرفته‌شده مستقیماً از `DOCX` دوباره استخراج می‌شوند.

هدف این مرحله این است که مطمئن شویم قرارداد تثبیت‌شده در Notebook 03 هنگام شروع Ingestion بدون تغییر باقی مانده است.

برای هر Edition پذیرفته‌شده موارد زیر دوباره محاسبه می‌شود:

- تعداد Paragraphهای معنی‌دار
- متن استخراج‌شده
- `text_sha256`
- Style هر Paragraph
- Heading Level
- `Heading Path`
- وضعیت قرارگیری Paragraph داخل Table
- `Stable Paragraph ID`

قرارداد Paragraph ID همان قرارداد پذیرفته‌شده Notebook 03 است:

`<edition_id>:p<meaningful-paragraph-sequence>`

در این مرحله سه Validation مهم انجام می‌شود:

1. تعداد Paragraphهای استخراج‌شده با مقدار ثبت‌شده در `book_manifest_v1.json` برابر باشد.
2. `text_sha256` استخراج جدید با Manifest برابر باشد.
3. تمام Paragraph IDها در هر Edition یکتا باشند.

این مرحله هنوز هیچ Artifact جدیدی روی Disk ایجاد نمی‌کند.

همچنین هیچ Chunking، Persian Normalization یا Deduplication انجام نمی‌شود؛ این موارد طبق Roadmap مراحل مستقل خود را دارند.

**معیار PASS:**

- هر سه Source بدون Error استخراج شوند.
- Paragraph Count هر سه Source با Manifest یکسان باشد.
- `text_sha256` هر سه Source با Manifest یکسان باشد.
- تمام Paragraph IDها یکتا باشند.
- هیچ Source خارج از `[3, 4, 6]` پردازش نشود.

</div>

In [2]:
# Code Cell 02

import hashlib
import re
import unicodedata
import zipfile
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"

def normalize_for_inventory_hash(text):
    text = unicodedata.normalize("NFC", text)
    return re.sub(r"\s+", " ", text).strip()

def heading_level_from_style(style):
    if not style:
        return None

    match = re.fullmatch(r"Heading(\d+)", style, flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

def extract_book_edition(path, work_id, edition_id):
    with zipfile.ZipFile(path) as archive:
        root = ET.fromstring(archive.read("word/document.xml"))

    table_paragraph_nodes = set()

    for table in root.findall(f".//{{{W_NS}}}tbl"):
        for paragraph in table.findall(f".//{{{W_NS}}}p"):
            table_paragraph_nodes.add(id(paragraph))

    paragraph_records = []
    heading_stack = {}

    for paragraph in root.findall(f".//{{{W_NS}}}p"):
        text = "".join(
            node.text or ""
            for node in paragraph.findall(f".//{{{W_NS}}}t")
        ).strip()

        if not text:
            continue

        style = None
        ppr = paragraph.find(f"{{{W_NS}}}pPr")

        if ppr is not None:
            style_node = ppr.find(f"{{{W_NS}}}pStyle")
            if style_node is not None:
                style = style_node.get(f"{{{W_NS}}}val")

        heading_level = heading_level_from_style(style)

        if heading_level is not None:
            heading_stack[heading_level] = text

            for level in list(heading_stack):
                if level > heading_level:
                    del heading_stack[level]

        paragraph_number = len(paragraph_records) + 1
        paragraph_id = f"{edition_id}:p{paragraph_number:06d}"

        paragraph_records.append({
            "work_id": work_id,
            "edition_id": edition_id,
            "paragraph_id": paragraph_id,
            "paragraph_number": paragraph_number,
            "text": text,
            "text_sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(),
            "style": style,
            "heading_level": heading_level,
            "heading_path": [
                heading_stack[level]
                for level in sorted(heading_stack)
            ],
            "is_heading": heading_level is not None,
            "in_table": id(paragraph) in table_paragraph_nodes,
        })

    full_text = "\n".join(
        record["text"]
        for record in paragraph_records
    )

    normalized_text = normalize_for_inventory_hash(full_text)

    return {
        "work_id": work_id,
        "edition_id": edition_id,
        "paragraphs": paragraph_records,
        "meaningful_paragraphs": len(paragraph_records),
        "text_sha256": hashlib.sha256(
            normalized_text.encode("utf-8")
        ).hexdigest(),
    }

accepted_book_extractions = {}

paragraph_count_mismatches = []
text_hash_mismatches = []
paragraph_id_collisions = []

for edition in accepted_editions:
    file_index = edition["file_index"]
    file_record = file_record_by_index[file_index]
    path = project_root / file_record["relative_path"]

    extraction = extract_book_edition(
        path=path,
        work_id=edition["work_id"],
        edition_id=edition["edition_id"],
    )

    accepted_book_extractions[file_index] = extraction

    paragraph_ids = [
        record["paragraph_id"]
        for record in extraction["paragraphs"]
    ]

    count_matches = (
        extraction["meaningful_paragraphs"]
        == file_record["meaningful_paragraphs"]
    )

    hash_matches = (
        extraction["text_sha256"]
        == file_record["text_sha256"]
    )

    ids_unique = (
        len(paragraph_ids)
        == len(set(paragraph_ids))
    )

    if not count_matches:
        paragraph_count_mismatches.append(file_index)

    if not hash_matches:
        text_hash_mismatches.append(file_index)

    if not ids_unique:
        paragraph_id_collisions.append(file_index)

    print("=" * 90)
    print(f"[{file_index}] {file_record['filename']}")
    print("Edition ID:", edition["edition_id"])
    print("Extracted paragraphs:", extraction["meaningful_paragraphs"])
    print("Manifest paragraphs:", file_record["meaningful_paragraphs"])
    print("Paragraph count match:", count_matches)
    print("Text SHA-256 match:", hash_matches)
    print("Unique paragraph IDs:", ids_unique)

validation_pass = (
    sorted(accepted_book_extractions) == [3, 4, 6]
    and not paragraph_count_mismatches
    and not text_hash_mismatches
    and not paragraph_id_collisions
)

print()
print("BOOK SOURCE RE-EXTRACTION SUMMARY")
print("-" * 80)
print("Sources processed:", sorted(accepted_book_extractions))
print("Paragraph count mismatches:", paragraph_count_mismatches)
print("Text hash mismatches:", text_hash_mismatches)
print("Paragraph ID collisions:", paragraph_id_collisions)
print()
print(
    "BOOK SOURCE RE-EXTRACTION GATE:",
    "PASS" if validation_pass else "FAIL",
)

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
Edition ID: book:adult-financial-literacy-course:fa-source
Extracted paragraphs: 1333
Manifest paragraphs: 1333
Paragraph count match: True
Text SHA-256 match: True
Unique paragraph IDs: True
[4] سواد مالی برای همه.docx
Edition ID: book:financial-literacy-for-all:fa-source
Extracted paragraphs: 871
Manifest paragraphs: 871
Paragraph count match: True
Text SHA-256 match: True
Unique paragraph IDs: True
[6] نوشتار آموزشی تربیت مربی سواد مالی.docx
Edition ID: book:trainer-training-financial-literacy:fa-source
Extracted paragraphs: 485
Manifest paragraphs: 485
Paragraph count match: True
Text SHA-256 match: True
Unique paragraph IDs: True

BOOK SOURCE RE-EXTRACTION SUMMARY
--------------------------------------------------------------------------------
Sources processed: [3, 4, 6]
Paragraph count mismatches: []
Text hash mismatches: []
Paragraph ID collisions: []

BOOK SOURCE RE-EXTRACTION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت ساختار Canonical اولیه برای Book Ingestion در حافظه

بازاستخراج مستقل سه منبع پذیرفته‌شده با موفقیت انجام شد:

`BOOK SOURCE RE-EXTRACTION GATE: PASS`

در این مرحله برای هر Edition پذیرفته‌شده یک Record ساختاریافته در حافظه ساخته می‌شود.

هدف این Recordها تبدیل فایل‌های Word به یک نمایش قابل‌ردیابی برای ادامه Pipeline کتاب است، بدون انجام Chunking، Deduplication یا Persian Normalization.

هر Record شامل سه لایه اطلاعات خواهد بود:

### هویت و Attribution

- `work_id`
- `edition_id`
- عنوان Work
- زبان
- `author`
- `knowledge_owner`
- `rights_status`

### Lineage فایل

- مسیر فایل اصلی
- `file_sha256`
- `text_sha256`
- تعداد Paragraphهای معنی‌دار

### محتوای استخراج‌شده

برای هر Paragraph:

- `paragraph_id`
- `paragraph_number`
- متن اصلی
- `text_sha256`
- `style`
- `heading_level`
- `heading_path`
- `is_heading`
- `in_table`

هیچ متن جدیدی ساخته یا بازنویسی نمی‌شود و متن همان محتوای استخراج‌شده از Source Edition است.

در این مرحله هنوز Artifact روی Disk ذخیره نمی‌شود. ابتدا باید ساختار Recordها و Lineage آن‌ها Validation شود.

**معیار PASS:**

- دقیقاً `3` Edition پذیرفته‌شده Record داشته باشند.
- هیچ Edition خارج از `[3, 4, 6]` وارد ساختار نشود.
- تمام Work ID و Edition IDها به Manifest معتبر متصل باشند.
- مجموع Paragraphها برابر با مجموع واقعی سه Source باشد.
- تمام Paragraph IDها در کل Corpus یکتا باشند.
- Attribution و Rights از Manifest بدون تغییر منتقل شوند.

</div>

In [3]:
# Code Cell 03

work_by_id = {
    work["work_id"]: work
    for work in works
}

book_ingestion_records = []

for edition in accepted_editions:
    file_index = edition["file_index"]
    work = work_by_id[edition["work_id"]]
    file_record = file_record_by_index[file_index]
    extraction = accepted_book_extractions[file_index]

    book_ingestion_records.append({
        "work_id": edition["work_id"],
        "edition_id": edition["edition_id"],
        "title": work["title"],
        "language": edition["language"],
        "author": work["author"],
        "knowledge_owner": work["knowledge_owner"],
        "rights_status": work["rights_status"],
        "rights_holder": work.get("rights_holder"),
        "rights_evidence_id": work.get("rights_evidence_id"),
        "allowed_uses": work.get("allowed_uses"),
        "source_file": {
            "file_index": file_index,
            "relative_path": file_record["relative_path"],
            "filename": file_record["filename"],
            "file_sha256": file_record["file_sha256"],
            "text_sha256": file_record["text_sha256"],
            "meaningful_paragraphs": file_record["meaningful_paragraphs"],
        },
        "paragraphs": extraction["paragraphs"],
    })

all_paragraph_ids = [
    paragraph["paragraph_id"]
    for record in book_ingestion_records
    for paragraph in record["paragraphs"]
]

total_paragraphs = sum(
    len(record["paragraphs"])
    for record in book_ingestion_records
)

record_file_indices = sorted(
    record["source_file"]["file_index"]
    for record in book_ingestion_records
)

edition_ids = [
    record["edition_id"]
    for record in book_ingestion_records
]

work_ids_valid = all(
    record["work_id"] in work_by_id
    for record in book_ingestion_records
)

rights_preserved = all(
    record["rights_status"] == "EVIDENCE_REQUIRED"
    for record in book_ingestion_records
)

expected_total_paragraphs = sum(
    file_record_by_index[index]["meaningful_paragraphs"]
    for index in [3, 4, 6]
)

structure_pass = (
    len(book_ingestion_records) == 3
    and record_file_indices == [3, 4, 6]
    and len(edition_ids) == len(set(edition_ids)) == 3
    and work_ids_valid
    and total_paragraphs == expected_total_paragraphs
    and len(all_paragraph_ids) == len(set(all_paragraph_ids))
    and rights_preserved
)

print("BOOK INGESTION RECORD STRUCTURE")
print("-" * 80)

for record in book_ingestion_records:
    print(record["edition_id"])
    print("    Work ID:", record["work_id"])
    print("    Title:", record["title"])
    print("    Source file index:", record["source_file"]["file_index"])
    print("    Paragraphs:", len(record["paragraphs"]))
    print("    Author:", record["author"])
    print("    Knowledge owner:", record["knowledge_owner"])
    print("    Rights status:", record["rights_status"])
    print()

print("INGESTION STRUCTURE SUMMARY")
print("-" * 80)
print("Edition records:", len(book_ingestion_records))
print("Source file indices:", record_file_indices)
print("Total paragraphs:", total_paragraphs)
print("Expected paragraphs:", expected_total_paragraphs)
print("Unique paragraph IDs:", len(set(all_paragraph_ids)))
print("Rights preserved:", rights_preserved)
print()
print(
    "BOOK INGESTION STRUCTURE GATE:",
    "PASS" if structure_pass else "FAIL",
)

BOOK INGESTION RECORD STRUCTURE
--------------------------------------------------------------------------------
book:adult-financial-literacy-course:fa-source
    Work ID: book:adult-financial-literacy-course
    Title: نوشتار آموزشی دوره آموزش سواد مالی بزرگسالان
    Source file index: 3
    Paragraphs: 1333
    Author: Dr. Komeil Roudi
    Knowledge owner: Dr. Komeil Roudi
    Rights status: EVIDENCE_REQUIRED

book:financial-literacy-for-all:fa-source
    Work ID: book:financial-literacy-for-all
    Title: سواد مالی برای همه
    Source file index: 4
    Paragraphs: 871
    Author: Dr. Komeil Roudi
    Knowledge owner: Dr. Komeil Roudi
    Rights status: EVIDENCE_REQUIRED

book:trainer-training-financial-literacy:fa-source
    Work ID: book:trainer-training-financial-literacy
    Title: نوشتار آموزشی تربیت مربی سواد مالی
    Source file index: 6
    Paragraphs: 485
    Author: Dr. Komeil Roudi
    Knowledge owner: Dr. Komeil Roudi
    Rights status: EVIDENCE_REQUIRED

INGESTION STRUC

<div style="direction: rtl; white-space: normal; line-height: 1;">

### Validation کیفیت پایه Paragraphهای Ingestion پیش از ذخیره Artifact

ساختار Canonical اولیه سه Edition پذیرفته‌شده با موفقیت ساخته شد:

`BOOK INGESTION STRUCTURE GATE: PASS`

در این مرحله هنوز هیچ Artifactی روی Disk نوشته نمی‌شود.

هدف این مرحله بررسی کیفیت پایه `2689` Paragraph استخراج‌شده است تا قبل از Persist کردن داده مطمئن شویم ساختار Ingestion سالم و قابل‌ردیابی است.

موارد بررسی‌شده:

- وجود تمام فیلدهای ضروری Paragraph
- نبود متن خالی
- یکتا بودن `paragraph_id`
- پیوسته بودن `paragraph_number` در هر Edition
- تطبیق `text_sha256` با متن واقعی Paragraph
- معتبر بودن `heading_path`
- معتبر بودن `is_heading` و `in_table`
- اندازه‌گیری تعداد Headingها و Paragraphهای داخل Table
- اندازه‌گیری Duplicate Text فقط به‌عنوان Evidence

در این مرحله Duplicate Text حذف یا اصلاح نمی‌شود.

Deduplication طبق Roadmap در:

`09_deduplication_and_quality.ipynb`

انجام خواهد شد.

همچنین هیچ Persian Normalization یا Chunking در این Cell انجام نمی‌شود.

**معیار PASS:**

- هیچ Paragraph فاقد فیلد ضروری نباشد.
- هیچ متن خالی وجود نداشته باشد.
- هیچ `paragraph_id` تکراری وجود نداشته باشد.
- شماره Paragraphها در هر Edition دقیقاً از `1` تا تعداد Paragraphهای همان Edition پیوسته باشند.
- هیچ `text_sha256` اشتباه وجود نداشته باشد.
- نوع داده فیلدهای ساختاری معتبر باشد.

</div>

In [4]:
# Code Cell 04

import hashlib
from collections import Counter

required_paragraph_fields = {
    "work_id",
    "edition_id",
    "paragraph_id",
    "paragraph_number",
    "text",
    "text_sha256",
    "style",
    "heading_level",
    "heading_path",
    "is_heading",
    "in_table",
}

missing_field_records = []
empty_text_records = []
hash_mismatches = []
invalid_heading_paths = []
invalid_boolean_fields = []
sequence_mismatches = []

all_paragraphs = [
    paragraph
    for record in book_ingestion_records
    for paragraph in record["paragraphs"]
]

for paragraph in all_paragraphs:
    missing_fields = (
        required_paragraph_fields
        - set(paragraph)
    )

    if missing_fields:
        missing_field_records.append({
            "paragraph_id": paragraph.get("paragraph_id"),
            "missing_fields": sorted(missing_fields),
        })

    if not paragraph.get("text", "").strip():
        empty_text_records.append(
            paragraph.get("paragraph_id")
        )

    actual_hash = hashlib.sha256(
        paragraph["text"].encode("utf-8")
    ).hexdigest()

    if actual_hash != paragraph["text_sha256"]:
        hash_mismatches.append(
            paragraph["paragraph_id"]
        )

    if not isinstance(paragraph["heading_path"], list):
        invalid_heading_paths.append(
            paragraph["paragraph_id"]
        )

    if (
        not isinstance(paragraph["is_heading"], bool)
        or not isinstance(paragraph["in_table"], bool)
    ):
        invalid_boolean_fields.append(
            paragraph["paragraph_id"]
        )

for record in book_ingestion_records:
    observed_numbers = [
        paragraph["paragraph_number"]
        for paragraph in record["paragraphs"]
    ]

    expected_numbers = list(
        range(1, len(record["paragraphs"]) + 1)
    )

    if observed_numbers != expected_numbers:
        sequence_mismatches.append(
            record["edition_id"]
        )

paragraph_ids = [
    paragraph["paragraph_id"]
    for paragraph in all_paragraphs
]

text_hash_counts = Counter(
    paragraph["text_sha256"]
    for paragraph in all_paragraphs
)

duplicate_text_groups = sum(
    1
    for count in text_hash_counts.values()
    if count > 1
)

duplicate_text_members = sum(
    count
    for count in text_hash_counts.values()
    if count > 1
)

heading_count = sum(
    paragraph["is_heading"]
    for paragraph in all_paragraphs
)

table_paragraph_count = sum(
    paragraph["in_table"]
    for paragraph in all_paragraphs
)

quality_gate_pass = (
    len(all_paragraphs) == 2689
    and len(paragraph_ids) == len(set(paragraph_ids))
    and not missing_field_records
    and not empty_text_records
    and not hash_mismatches
    and not invalid_heading_paths
    and not invalid_boolean_fields
    and not sequence_mismatches
)

print("BOOK PARAGRAPH INGESTION QUALITY")
print("-" * 80)
print("Paragraphs:", len(all_paragraphs))
print("Unique paragraph IDs:", len(set(paragraph_ids)))
print("Missing-field records:", len(missing_field_records))
print("Empty-text records:", len(empty_text_records))
print("Text hash mismatches:", len(hash_mismatches))
print("Invalid heading paths:", len(invalid_heading_paths))
print("Invalid boolean fields:", len(invalid_boolean_fields))
print("Paragraph sequence mismatches:", len(sequence_mismatches))
print("Heading paragraphs:", heading_count)
print("Paragraphs inside tables:", table_paragraph_count)
print("Duplicate-text groups:", duplicate_text_groups)
print("Duplicate-text member paragraphs:", duplicate_text_members)
print()
print(
    "BOOK PARAGRAPH QUALITY GATE:",
    "PASS" if quality_gate_pass else "FAIL",
)

BOOK PARAGRAPH INGESTION QUALITY
--------------------------------------------------------------------------------
Paragraphs: 2689
Unique paragraph IDs: 2689
Missing-field records: 0
Empty-text records: 0
Text hash mismatches: 0
Invalid heading paths: 0
Invalid boolean fields: 0
Paragraph sequence mismatches: 0
Heading paragraphs: 180
Paragraphs inside tables: 1086
Duplicate-text groups: 104
Duplicate-text member paragraphs: 568

BOOK PARAGRAPH QUALITY GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت Artifact ساختاریافته Book Ingestion

Validation کیفیت پایه تمام `2689` Paragraph با موفقیت انجام شد:

`BOOK PARAGRAPH QUALITY GATE: PASS`

اکنون داده استخراج‌شده سه Edition پذیرفته‌شده می‌تواند به‌عنوان اولین Artifact رسمی Book Ingestion روی Disk ثبت شود.

Artifact این مرحله:

`data/phase2/interim/book_ingested_v1.json`

این فایل یک Snapshot ساختاریافته از Sourceهای پذیرفته‌شده است و شامل:

- هویت Work و Edition
- Attribution
- وضعیت Rights
- Lineage فایل اصلی
- Hash فایل و متن
- Paragraphهای استخراج‌شده
- Stable Paragraph ID
- Heading Structure
- Table membership

خواهد بود.

برای حفظ Lineage، `SHA-256` فایل:

`data/phase2/manifests/book_manifest_v1.json`

نیز در Artifact ثبت می‌شود تا مشخص باشد این Ingestion دقیقاً از کدام نسخه Manifest ساخته شده است.

این Artifact هنوز داده نرمال‌شده یا Deduplicated نیست.

بنابراین:

- متن Paragraphها تغییر نمی‌کند.
- Duplicateها حذف نمی‌شوند.
- Chunk ساخته نمی‌شود.
- Persian Normalization انجام نمی‌شود.

Artifact به‌صورت Idempotent ساخته می‌شود:

`Missing → CREATED`

`Existing + Identical → REUSED_IDENTICAL`

`Existing + Different → FAIL`

هیچ Artifact موجود متفاوتی به‌صورت Silent overwrite نخواهد شد.

**معیار PASS:**

- Artifact شامل دقیقاً `3` Edition باشد.
- مجموع Paragraphها دقیقاً `2689` باشد.
- Source Manifest Hash ثبت شود.
- تصمیم‌های Rights و Attribution حفظ شوند.
- Artifact موجود متفاوت overwrite نشود.

</div>

In [5]:
# Code Cell 05

import hashlib
import json

interim_path = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "book_ingested_v1.json"
)

manifest_sha256 = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

book_ingested_artifact = {
    "artifact_type": "phase2_book_ingestion",
    "artifact_version": "v1",
    "schema_version": "book_ingestion_v1",
    "source_type": "book",
    "source_manifest": {
        "relative_path": str(
            manifest_path.relative_to(project_root)
        ),
        "sha256": manifest_sha256,
        "manifest_version": book_manifest["manifest_version"],
    },
    "edition_count": len(book_ingestion_records),
    "paragraph_count": len(all_paragraphs),
    "edition_records": book_ingestion_records,
    "processing_state": {
        "source_extraction": "COMPLETED",
        "persian_normalization": "NOT_PERFORMED",
        "deduplication": "NOT_PERFORMED",
        "chunking": "NOT_PERFORMED",
    },
}

interim_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if interim_path.exists():
    with interim_path.open("r", encoding="utf-8") as file:
        existing_artifact = json.load(file)

    if existing_artifact != book_ingested_artifact:
        raise ValueError(
            "Existing book_ingested_v1.json differs from the "
            "deterministic candidate. Refusing to overwrite."
        )

    artifact_action = "REUSED_IDENTICAL"

else:
    with interim_path.open("w", encoding="utf-8") as file:
        json.dump(
            book_ingested_artifact,
            file,
            ensure_ascii=False,
            indent=2,
        )

    artifact_action = "CREATED"

artifact_creation_pass = (
    interim_path.exists()
    and book_ingested_artifact["edition_count"] == 3
    and book_ingested_artifact["paragraph_count"] == 2689
    and bool(manifest_sha256)
    and book_ingested_artifact["processing_state"]["chunking"]
    == "NOT_PERFORMED"
    and book_ingested_artifact["processing_state"]["deduplication"]
    == "NOT_PERFORMED"
    and artifact_action in {"CREATED", "REUSED_IDENTICAL"}
)

print("BOOK INGESTION INTERIM ARTIFACT")
print("-" * 80)
print("Path:", interim_path.relative_to(project_root))
print("Action:", artifact_action)
print("Artifact version:", book_ingested_artifact["artifact_version"])
print("Schema version:", book_ingested_artifact["schema_version"])
print("Source manifest SHA-256:", manifest_sha256)
print("Editions:", book_ingested_artifact["edition_count"])
print("Paragraphs:", book_ingested_artifact["paragraph_count"])
print(
    "Persian normalization:",
    book_ingested_artifact["processing_state"]["persian_normalization"],
)
print(
    "Deduplication:",
    book_ingested_artifact["processing_state"]["deduplication"],
)
print(
    "Chunking:",
    book_ingested_artifact["processing_state"]["chunking"],
)
print()
print(
    "BOOK INGESTION ARTIFACT CREATION GATE:",
    "PASS" if artifact_creation_pass else "FAIL",
)

BOOK INGESTION INTERIM ARTIFACT
--------------------------------------------------------------------------------
Path: data/phase2/interim/book_ingested_v1.json
Action: REUSED_IDENTICAL
Artifact version: v1
Schema version: book_ingestion_v1
Source manifest SHA-256: 6e5be92bff65d55a7255be3884cc65ac8978b940b8039aaed129c23006a63276
Editions: 3
Paragraphs: 2689
Persian normalization: NOT_PERFORMED
Deduplication: NOT_PERFORMED
Chunking: NOT_PERFORMED

BOOK INGESTION ARTIFACT CREATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### Independent Re-validation فایل `book_ingested_v1.json`

Artifact اولیه Book Ingestion با موفقیت ساخته شد:

`BOOK INGESTION ARTIFACT CREATION GATE: PASS`

اما ساخته‌شدن فایل به‌تنهایی برای پذیرش آن کافی نیست.

در این مرحله فایل:

`data/phase2/interim/book_ingested_v1.json`

مستقیماً از Disk دوباره خوانده می‌شود و مستقل از Variableهای ساخته‌شده در Cell قبلی Validation خواهد شد.

موارد بررسی‌شده:

- نوع و Version Artifact
- Schema Version
- تعداد Editionها
- تعداد کل Paragraphها
- یکتا بودن Edition IDها
- یکتا بودن تمام Paragraph IDها
- پیوستگی Paragraph Number در هر Edition
- صحت `text_sha256` تمام Paragraphها
- وجود و تطبیق Source Manifest
- تطبیق `SHA-256` Manifest ثبت‌شده با فایل واقعی
- تطبیق Lineage هر Edition با `book_manifest_v1.json`
- تطبیق `file_sha256`، `text_sha256` و Paragraph Count با Manifest
- حفظ Attribution و Rights
- تأیید اینکه Persian Normalization، Deduplication و Chunking هنوز انجام نشده‌اند

این Cell هیچ فایل یا Artifactی را تغییر نمی‌دهد.

**معیار PASS:**

تمام Identity، Content Hash و Lineage Contractهای Artifact ذخیره‌شده باید بدون mismatch تأیید شوند.

</div>

In [6]:
# Code Cell 06

import hashlib
import json

with interim_path.open("r", encoding="utf-8") as file:
    disk_artifact = json.load(file)

disk_edition_records = disk_artifact.get("edition_records", [])

disk_all_paragraphs = [
    paragraph
    for record in disk_edition_records
    for paragraph in record.get("paragraphs", [])
]

disk_edition_ids = [
    record.get("edition_id")
    for record in disk_edition_records
]

disk_paragraph_ids = [
    paragraph.get("paragraph_id")
    for paragraph in disk_all_paragraphs
]

paragraph_hash_mismatches = []
paragraph_sequence_mismatches = []
lineage_mismatches = []

for record in disk_edition_records:
    observed_numbers = [
        paragraph.get("paragraph_number")
        for paragraph in record.get("paragraphs", [])
    ]

    expected_numbers = list(
        range(1, len(record.get("paragraphs", [])) + 1)
    )

    if observed_numbers != expected_numbers:
        paragraph_sequence_mismatches.append(
            record.get("edition_id")
        )

    for paragraph in record.get("paragraphs", []):
        actual_hash = hashlib.sha256(
            paragraph.get("text", "").encode("utf-8")
        ).hexdigest()

        if actual_hash != paragraph.get("text_sha256"):
            paragraph_hash_mismatches.append(
                paragraph.get("paragraph_id")
            )

manifest_file_by_edition = {
    edition["edition_id"]: file_record_by_index[edition["file_index"]]
    for edition in accepted_editions
}

for record in disk_edition_records:
    edition_id = record.get("edition_id")
    source_file = record.get("source_file", {})
    manifest_file = manifest_file_by_edition.get(edition_id)

    if manifest_file is None:
        lineage_mismatches.append({
            "edition_id": edition_id,
            "reason": "Edition not found in accepted manifest sources",
        })
        continue

    checks = {
        "relative_path": (
            source_file.get("relative_path")
            == manifest_file.get("relative_path")
        ),
        "file_sha256": (
            source_file.get("file_sha256")
            == manifest_file.get("file_sha256")
        ),
        "text_sha256": (
            source_file.get("text_sha256")
            == manifest_file.get("text_sha256")
        ),
        "meaningful_paragraphs": (
            source_file.get("meaningful_paragraphs")
            == manifest_file.get("meaningful_paragraphs")
        ),
    }

    if not all(checks.values()):
        lineage_mismatches.append({
            "edition_id": edition_id,
            "failed_checks": [
                name
                for name, passed in checks.items()
                if not passed
            ],
        })

actual_manifest_sha256 = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

stored_manifest_sha256 = (
    disk_artifact
    .get("source_manifest", {})
    .get("sha256")
)

processing_state = disk_artifact.get(
    "processing_state",
    {}
)

independent_artifact_pass = (
    disk_artifact.get("artifact_type")
    == "phase2_book_ingestion"
    and disk_artifact.get("artifact_version") == "v1"
    and disk_artifact.get("schema_version")
    == "book_ingestion_v1"
    and len(disk_edition_records) == 3
    and len(set(disk_edition_ids)) == 3
    and len(disk_all_paragraphs) == 2689
    and len(disk_paragraph_ids)
    == len(set(disk_paragraph_ids))
    and not paragraph_hash_mismatches
    and not paragraph_sequence_mismatches
    and not lineage_mismatches
    and stored_manifest_sha256 == actual_manifest_sha256
    and processing_state.get("persian_normalization")
    == "NOT_PERFORMED"
    and processing_state.get("deduplication")
    == "NOT_PERFORMED"
    and processing_state.get("chunking")
    == "NOT_PERFORMED"
)

print("BOOK INGESTION ARTIFACT INDEPENDENT VALIDATION")
print("-" * 80)
print("Artifact type:", disk_artifact.get("artifact_type"))
print("Artifact version:", disk_artifact.get("artifact_version"))
print("Schema version:", disk_artifact.get("schema_version"))
print("Editions:", len(disk_edition_records))
print("Paragraphs:", len(disk_all_paragraphs))
print("Unique Edition IDs:", len(set(disk_edition_ids)))
print("Unique Paragraph IDs:", len(set(disk_paragraph_ids)))
print("Paragraph hash mismatches:", len(paragraph_hash_mismatches))
print("Paragraph sequence mismatches:", len(paragraph_sequence_mismatches))
print("Lineage mismatches:", len(lineage_mismatches))
print(
    "Source manifest hash match:",
    stored_manifest_sha256 == actual_manifest_sha256,
)
print(
    "Persian normalization:",
    processing_state.get("persian_normalization"),
)
print(
    "Deduplication:",
    processing_state.get("deduplication"),
)
print(
    "Chunking:",
    processing_state.get("chunking"),
)
print()
print(
    "BOOK INGESTION ARTIFACT INDEPENDENT VALIDATION GATE:",
    "PASS" if independent_artifact_pass else "FAIL",
)

BOOK INGESTION ARTIFACT INDEPENDENT VALIDATION
--------------------------------------------------------------------------------
Artifact type: phase2_book_ingestion
Artifact version: v1
Schema version: book_ingestion_v1
Editions: 3
Paragraphs: 2689
Unique Edition IDs: 3
Unique Paragraph IDs: 2689
Paragraph hash mismatches: 0
Paragraph sequence mismatches: 0
Lineage mismatches: 0
Source manifest hash match: True
Persian normalization: NOT_PERFORMED
Deduplication: NOT_PERFORMED
Chunking: NOT_PERFORMED

BOOK INGESTION ARTIFACT INDEPENDENT VALIDATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ممیزی پوشش ساختار DOCX و محتوای خارج از Paragraphهای اصلی

Artifact ذخیره‌شده Book Ingestion به‌صورت مستقل Validation شد:

`BOOK INGESTION ARTIFACT INDEPENDENT VALIDATION GATE: PASS`

اما صحت Paragraphهای استخراج‌شده به‌تنهایی ثابت نمی‌کند که تمام محتوای بالقوه مهم فایل Word پوشش داده شده است.

یک فایل `DOCX` می‌تواند علاوه بر متن اصلی Document شامل اجزای دیگری نیز باشد، مانند:

- تصویر و Diagram
- Header
- Footer
- Footnote
- Endnote
- Comment
- Textbox یا Drawing
- Embedded Object

اگر چنین محتوایی وجود داشته باشد، نباید بدون بررسی از Pipeline حذف شود.

در این مرحله Package داخلی سه Source پذیرفته‌شده `[3, 4, 6]` ممیزی می‌شود تا مشخص شود چه اجزای غیرمتنی یا خارج از Body اصلی وجود دارند.

برای هر فایل موارد زیر اندازه‌گیری می‌شود:

- تعداد فایل‌های Media
- تعداد Drawingها
- تعداد Headerها
- تعداد Footerها
- وجود Footnote
- وجود Endnote
- وجود Comment
- تعداد Embedded Objectها
- تعداد Hyperlinkها

این مرحله فقط `Discovery / Inspection` است.

هیچ تصویر یا محتوای جانبی استخراج، تبدیل یا حذف نمی‌شود و هیچ Artifact فعلی تغییر نمی‌کند.

وجود این اجزا الزاماً به معنی FAIL نیست؛ هدف این است که قبل از Final Acceptance مشخص شود آیا Ingestion فعلی فقط Text Corpus را پوشش می‌دهد یا محتوای دیگری نیز نیازمند سیاست مستقل است.

**معیار PASS:**

هر سه Source باید بدون Error ممیزی شوند و وضعیت اجزای جانبی آن‌ها به‌صورت صریح مشخص شود.

</div>

In [7]:
# Code Cell 07

import zipfile
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
A_NS = "http://schemas.openxmlformats.org/drawingml/2006/main"
R_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"

def inspect_docx_package(path):
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()

        document_root = ET.fromstring(
            archive.read("word/document.xml")
        )

        media_files = [
            name
            for name in names
            if name.startswith("word/media/")
            and not name.endswith("/")
        ]

        headers = [
            name
            for name in names
            if name.startswith("word/header")
            and name.endswith(".xml")
        ]

        footers = [
            name
            for name in names
            if name.startswith("word/footer")
            and name.endswith(".xml")
        ]

        embedded_objects = [
            name
            for name in names
            if name.startswith("word/embeddings/")
            and not name.endswith("/")
        ]

        has_footnotes = "word/footnotes.xml" in names
        has_endnotes = "word/endnotes.xml" in names
        has_comments = "word/comments.xml" in names

        drawings = document_root.findall(
            f".//{{{W_NS}}}drawing"
        )

        hyperlinks = document_root.findall(
            f".//{{{W_NS}}}hyperlink"
        )

        drawing_text_nodes = document_root.findall(
            f".//{{{A_NS}}}t"
        )

        drawing_text = [
            node.text.strip()
            for node in drawing_text_nodes
            if node.text and node.text.strip()
        ]

    return {
        "media_files": media_files,
        "headers": headers,
        "footers": footers,
        "embedded_objects": embedded_objects,
        "has_footnotes": has_footnotes,
        "has_endnotes": has_endnotes,
        "has_comments": has_comments,
        "drawing_count": len(drawings),
        "hyperlink_count": len(hyperlinks),
        "drawing_text": drawing_text,
    }

package_audit_results = {}

for file_index in [3, 4, 6]:
    file_record = file_record_by_index[file_index]
    path = project_root / file_record["relative_path"]

    result = inspect_docx_package(path)
    package_audit_results[file_index] = result

    print("=" * 90)
    print(f"[{file_index}] {file_record['filename']}")
    print("Media files:", len(result["media_files"]))
    print("Drawings:", result["drawing_count"])
    print("Drawing text nodes:", len(result["drawing_text"]))
    print("Headers:", len(result["headers"]))
    print("Footers:", len(result["footers"]))
    print("Footnotes:", result["has_footnotes"])
    print("Endnotes:", result["has_endnotes"])
    print("Comments:", result["has_comments"])
    print("Embedded objects:", len(result["embedded_objects"]))
    print("Hyperlinks:", result["hyperlink_count"])

    if result["drawing_text"]:
        print("Drawing text samples:")
        for text in result["drawing_text"][:10]:
            print(" -", text[:200])

    print()

package_audit_pass = (
    sorted(package_audit_results) == [3, 4, 6]
)

print("BOOK DOCX PACKAGE AUDIT SUMMARY")
print("-" * 80)
print("Sources audited:", sorted(package_audit_results))
print(
    "Total media files:",
    sum(len(result["media_files"]) for result in package_audit_results.values()),
)
print(
    "Total drawings:",
    sum(result["drawing_count"] for result in package_audit_results.values()),
)
print(
    "Total drawing text nodes:",
    sum(len(result["drawing_text"]) for result in package_audit_results.values()),
)
print()
print(
    "BOOK DOCX PACKAGE AUDIT GATE:",
    "PASS" if package_audit_pass else "FAIL",
)

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
Media files: 2
Drawings: 8
Drawing text nodes: 0
Headers: 0
Footers: 2
Footnotes: True
Endnotes: True
Comments: False
Embedded objects: 1
Hyperlinks: 150

[4] سواد مالی برای همه.docx
Media files: 1
Drawings: 13
Drawing text nodes: 0
Headers: 0
Footers: 1
Footnotes: True
Endnotes: True
Comments: False
Embedded objects: 8
Hyperlinks: 0

[6] نوشتار آموزشی تربیت مربی سواد مالی.docx
Media files: 2
Drawings: 3
Drawing text nodes: 0
Headers: 3
Footers: 3
Footnotes: True
Endnotes: True
Comments: False
Embedded objects: 0
Hyperlinks: 74

BOOK DOCX PACKAGE AUDIT SUMMARY
--------------------------------------------------------------------------------
Sources audited: [3, 4, 6]
Total media files: 5
Total drawings: 24
Total drawing text nodes: 0

BOOK DOCX PACKAGE AUDIT GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی محتوای واقعی خارج از Body اصلی DOCX

ممیزی Package سه Source پذیرفته‌شده با موفقیت انجام شد:

`BOOK DOCX PACKAGE AUDIT GATE: PASS`

با این حال در هر سه فایل اجزایی خارج از Paragraphهای اصلی Document مشاهده شد؛ از جمله:

- Footer
- Footnote / Endnote
- Media
- Drawing
- Embedded Object

وجود فایل‌هایی مانند `footnotes.xml` و `endnotes.xml` به‌تنهایی ثابت نمی‌کند که Footnote یا Endnote واقعی و محتوایی وجود دارد؛ Word می‌تواند Entryهای سیستمی مانند Separator را نیز در این فایل‌ها نگهداری کند.

بنابراین در این مرحله متن واقعی بخش‌های خارج از Body بررسی می‌شود.

موارد بررسی:

- متن Headerها
- متن Footerها
- Footnoteهای واقعی با حذف Entryهای سیستمی
- Endnoteهای واقعی با حذف Entryهای سیستمی
- نام و نوع Embedded Objectها
- Alt Text یا Description مربوط به Drawingها در صورت وجود

هدف این است که مشخص شود آیا Artifact فعلی `book_ingested_v1.json` محتوای متنی معناداری را نادیده گرفته است یا خیر.

این مرحله هنوز Artifact موجود را تغییر نمی‌دهد.

اگر محتوای معناداری خارج از Body پیدا شود، قبل از هر Repair باید نوع و نقش آن بررسی شود.

**معیار PASS:**

- هر سه Source بدون Error بررسی شوند.
- محتوای واقعی Header/Footer/Footnote/Endnote از Entryهای صرفاً سیستمی تفکیک شود.
- Embedded Objectها و Drawing Metadata بدون حدس شناسایی شوند.
- مشخص شود آیا Text Ingestion فعلی نیازمند بررسی تکمیلی است یا خیر.

</div>

In [8]:
# Code Cell 08

import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
WP_NS = "http://schemas.openxmlformats.org/drawingml/2006/wordprocessingDrawing"

def xml_text(root):
    texts = [
        node.text
        for node in root.findall(f".//{{{W_NS}}}t")
        if node.text and node.text.strip()
    ]
    return " ".join(text.strip() for text in texts).strip()

def inspect_non_body_content(path):
    result = {
        "headers": [],
        "footers": [],
        "footnotes": [],
        "endnotes": [],
        "embedded_objects": [],
        "drawing_metadata": [],
    }

    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()

        for name in sorted(names):
            if (
                name.startswith("word/header")
                and name.endswith(".xml")
            ):
                root = ET.fromstring(archive.read(name))
                text = xml_text(root)

                if text:
                    result["headers"].append({
                        "part": name,
                        "text": text,
                    })

            if (
                name.startswith("word/footer")
                and name.endswith(".xml")
            ):
                root = ET.fromstring(archive.read(name))
                text = xml_text(root)

                if text:
                    result["footers"].append({
                        "part": name,
                        "text": text,
                    })

        if "word/footnotes.xml" in names:
            root = ET.fromstring(
                archive.read("word/footnotes.xml")
            )

            for note in root.findall(f"{{{W_NS}}}footnote"):
                note_id = note.get(f"{{{W_NS}}}id")

                try:
                    numeric_id = int(note_id)
                except (TypeError, ValueError):
                    numeric_id = None

                text = xml_text(note)

                if (
                    numeric_id is not None
                    and numeric_id > 0
                    and text
                ):
                    result["footnotes"].append({
                        "id": numeric_id,
                        "text": text,
                    })

        if "word/endnotes.xml" in names:
            root = ET.fromstring(
                archive.read("word/endnotes.xml")
            )

            for note in root.findall(f"{{{W_NS}}}endnote"):
                note_id = note.get(f"{{{W_NS}}}id")

                try:
                    numeric_id = int(note_id)
                except (TypeError, ValueError):
                    numeric_id = None

                text = xml_text(note)

                if (
                    numeric_id is not None
                    and numeric_id > 0
                    and text
                ):
                    result["endnotes"].append({
                        "id": numeric_id,
                        "text": text,
                    })

        result["embedded_objects"] = sorted(
            Path(name).name
            for name in names
            if name.startswith("word/embeddings/")
            and not name.endswith("/")
        )

        document_root = ET.fromstring(
            archive.read("word/document.xml")
        )

        for node in document_root.findall(
            f".//{{{WP_NS}}}docPr"
        ):
            metadata = {
                "name": node.get("name"),
                "title": node.get("title"),
                "description": node.get("descr"),
            }

            if any(metadata.values()):
                result["drawing_metadata"].append(metadata)

    return result

non_body_results = {}

for file_index in [3, 4, 6]:
    file_record = file_record_by_index[file_index]
    path = project_root / file_record["relative_path"]

    result = inspect_non_body_content(path)
    non_body_results[file_index] = result

    print("=" * 90)
    print(f"[{file_index}] {file_record['filename']}")

    print("Meaningful headers:", len(result["headers"]))
    for item in result["headers"][:10]:
        print(" -", item["part"], ":", item["text"][:300])

    print("Meaningful footers:", len(result["footers"]))
    for item in result["footers"][:10]:
        print(" -", item["part"], ":", item["text"][:300])

    print("Real footnotes:", len(result["footnotes"]))
    for item in result["footnotes"][:10]:
        print(f" - [{item['id']}]", item["text"][:300])

    print("Real endnotes:", len(result["endnotes"]))
    for item in result["endnotes"][:10]:
        print(f" - [{item['id']}]", item["text"][:300])

    print(
        "Embedded objects:",
        len(result["embedded_objects"]),
        result["embedded_objects"],
    )

    print(
        "Drawing metadata records:",
        len(result["drawing_metadata"]),
    )

    for item in result["drawing_metadata"][:10]:
        print(
            " -",
            f"name={item['name']!r}",
            f"title={item['title']!r}",
            f"description={item['description']!r}",
        )

    print()

meaningful_non_body_text_count = sum(
    len(result["headers"])
    + len(result["footers"])
    + len(result["footnotes"])
    + len(result["endnotes"])
    for result in non_body_results.values()
)

inspection_pass = (
    sorted(non_body_results) == [3, 4, 6]
)

print("NON-BODY CONTENT SUMMARY")
print("-" * 80)
print(
    "Meaningful non-body text records:",
    meaningful_non_body_text_count,
)
print(
    "Embedded objects:",
    sum(
        len(result["embedded_objects"])
        for result in non_body_results.values()
    ),
)
print(
    "Drawing metadata records:",
    sum(
        len(result["drawing_metadata"])
        for result in non_body_results.values()
    ),
)
print()
print(
    "BOOK NON-BODY CONTENT INSPECTION GATE:",
    "PASS" if inspection_pass else "FAIL",
)

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
Meaningful headers: 0
Meaningful footers: 2
 - word/footer1.xml : 5
 - word/footer2.xml : 3
Real footnotes: 0
Real endnotes: 0
Embedded objects: 1 ['Microsoft_Excel_Worksheet.xlsx']
Drawing metadata records: 8
 - name='Picture 2' title=None description=None
 - name='Picture 8' title=None description=None
 - name='Chart 4' title=None description=None
 - name='Chart 5' title=None description=None
 - name='Chart 7' title=None description=None
 - name='Chart 8' title=None description=None
 - name='Chart 10' title=None description=None
 - name='Chart 6' title=None description=None

[4] سواد مالی برای همه.docx
Meaningful headers: 0
Meaningful footers: 1
 - word/footer1.xml : 49
Real footnotes: 2
 - [1] برای همین است که پول دادن به نیازمندان، معمولاً مشکلات مالی آن‌ها را حل نمی‌کند!
 - [2] ام.بی.تی.آی. یکی از معتبرترین و شناحته شده‌ترین ابزارهای شخصیت‌شناسی است؛ سالانه دو میلیون نفر از این ابزار در کاریابی بهره می‌ برند. مبتنی بر این ابزا

<div style="direction: rtl; white-space: normal; line-height: 1;">

### تحلیل Lineage محتوای جانبی و Footnote Referenceها

ممیزی محتوای خارج از Body نشان داد که Text Ingestion فعلی هنوز تمام اطلاعات Source را پوشش نمی‌دهد.

شواهد واقعی:

- فایل `[4]` دارای `2` Footnote واقعی است.
- فایل `[6]` دارای `10` Footnote واقعی است.
- فایل `[3]` Footnote واقعی ندارد.
- در مجموع `9` Embedded Object وجود دارد.
- `24` Drawing نیز در Sourceها وجود دارد.

بنابراین قبل از هر Repair باید مشخص شود:

1. Footnoteهای استخراج‌شده واقعاً از Body اصلی Reference شده‌اند یا خیر.
2. هیچ Footnote Reference شکسته یا Footnote بدون Reference وجود نداشته باشد.
3. Footerهای مشاهده‌شده فقط Pagination هستند یا محتوای دانشی دارند.
4. نوع، نام و اندازه Embedded Objectها دقیقاً چیست.
5. Mediaها و Embedded Objectها باید برای بررسی بعدی Fingerprint شوند.

در این مرحله هنوز Artifact `book_ingested_v1.json` تغییر نمی‌کند.

قاعده:

`DISCOVER → VALIDATE → THEN REPAIR`

Footnoteها نیز بدون بررسی به Paragraphهای اصلی تبدیل نمی‌شوند، زیرا Citation Contract فعلی بر پایه Paragraph ID طراحی شده و Locator مربوط به Note باید صریح و قابل‌ردیابی تعریف شود.

**معیار PASS:**

- Footnote Referenceهای Body با Footnoteهای واقعی تطبیق داده شوند.
- Missing یا Orphan Noteها مشخص شوند.
- Footerها از نظر عددی بودن بررسی شوند.
- تمام Embedded Objectها و Mediaها با نام، اندازه و Hash شناسایی شوند.

</div>

In [9]:
# Code Cell 09

import hashlib
import re
import zipfile
from pathlib import Path
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"

def inspect_note_and_binary_lineage(path):
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()

        document_root = ET.fromstring(
            archive.read("word/document.xml")
        )

        footnote_reference_ids = [
            int(node.get(f"{{{W_NS}}}id"))
            for node in document_root.findall(
                f".//{{{W_NS}}}footnoteReference"
            )
            if node.get(f"{{{W_NS}}}id") is not None
        ]

        endnote_reference_ids = [
            int(node.get(f"{{{W_NS}}}id"))
            for node in document_root.findall(
                f".//{{{W_NS}}}endnoteReference"
            )
            if node.get(f"{{{W_NS}}}id") is not None
        ]

        real_footnote_ids = []

        if "word/footnotes.xml" in names:
            footnote_root = ET.fromstring(
                archive.read("word/footnotes.xml")
            )

            for note in footnote_root.findall(
                f"{{{W_NS}}}footnote"
            ):
                note_id = note.get(f"{{{W_NS}}}id")

                try:
                    note_id = int(note_id)
                except (TypeError, ValueError):
                    continue

                text = " ".join(
                    node.text.strip()
                    for node in note.findall(
                        f".//{{{W_NS}}}t"
                    )
                    if node.text and node.text.strip()
                )

                if note_id > 0 and text:
                    real_footnote_ids.append(note_id)

        real_endnote_ids = []

        if "word/endnotes.xml" in names:
            endnote_root = ET.fromstring(
                archive.read("word/endnotes.xml")
            )

            for note in endnote_root.findall(
                f"{{{W_NS}}}endnote"
            ):
                note_id = note.get(f"{{{W_NS}}}id")

                try:
                    note_id = int(note_id)
                except (TypeError, ValueError):
                    continue

                text = " ".join(
                    node.text.strip()
                    for node in note.findall(
                        f".//{{{W_NS}}}t"
                    )
                    if node.text and node.text.strip()
                )

                if note_id > 0 and text:
                    real_endnote_ids.append(note_id)

        footer_records = []

        for name in sorted(names):
            if (
                name.startswith("word/footer")
                and name.endswith(".xml")
            ):
                root = ET.fromstring(archive.read(name))

                text = " ".join(
                    node.text.strip()
                    for node in root.findall(
                        f".//{{{W_NS}}}t"
                    )
                    if node.text and node.text.strip()
                )

                if text:
                    footer_records.append({
                        "part": name,
                        "text": text,
                        "numeric_only": bool(
                            re.fullmatch(r"\d+", text)
                        ),
                    })

        binary_records = []

        for name in sorted(names):
            if not (
                name.startswith("word/embeddings/")
                or name.startswith("word/media/")
            ):
                continue

            if name.endswith("/"):
                continue

            data = archive.read(name)

            binary_records.append({
                "part": name,
                "filename": Path(name).name,
                "size_bytes": len(data),
                "sha256": hashlib.sha256(data).hexdigest(),
                "category": (
                    "EMBEDDED_OBJECT"
                    if name.startswith("word/embeddings/")
                    else "MEDIA"
                ),
            })

    footnote_reference_set = set(footnote_reference_ids)
    footnote_definition_set = set(real_footnote_ids)

    endnote_reference_set = set(endnote_reference_ids)
    endnote_definition_set = set(real_endnote_ids)

    return {
        "footnote_reference_ids": footnote_reference_ids,
        "real_footnote_ids": real_footnote_ids,
        "missing_footnote_definitions": sorted(
            footnote_reference_set - footnote_definition_set
        ),
        "orphan_footnotes": sorted(
            footnote_definition_set - footnote_reference_set
        ),
        "endnote_reference_ids": endnote_reference_ids,
        "real_endnote_ids": real_endnote_ids,
        "missing_endnote_definitions": sorted(
            endnote_reference_set - endnote_definition_set
        ),
        "orphan_endnotes": sorted(
            endnote_definition_set - endnote_reference_set
        ),
        "footers": footer_records,
        "binaries": binary_records,
    }

non_body_lineage_results = {}

for file_index in [3, 4, 6]:
    file_record = file_record_by_index[file_index]
    path = project_root / file_record["relative_path"]

    result = inspect_note_and_binary_lineage(path)
    non_body_lineage_results[file_index] = result

    print("=" * 100)
    print(f"[{file_index}] {file_record['filename']}")

    print("Footnote references:", result["footnote_reference_ids"])
    print("Real footnote definitions:", result["real_footnote_ids"])
    print(
        "Missing footnote definitions:",
        result["missing_footnote_definitions"],
    )
    print("Orphan footnotes:", result["orphan_footnotes"])

    print("Endnote references:", result["endnote_reference_ids"])
    print("Real endnote definitions:", result["real_endnote_ids"])
    print(
        "Missing endnote definitions:",
        result["missing_endnote_definitions"],
    )
    print("Orphan endnotes:", result["orphan_endnotes"])

    print("\nFooters:")
    if result["footers"]:
        for footer in result["footers"]:
            print(
                f" - {footer['part']} "
                f"| text={footer['text']!r} "
                f"| numeric_only={footer['numeric_only']}"
            )
    else:
        print(" - NONE")

    print("\nBinary package parts:")
    if result["binaries"]:
        for item in result["binaries"]:
            print(
                f" - {item['category']} "
                f"| {item['filename']} "
                f"| {item['size_bytes']} bytes "
                f"| SHA-256 {item['sha256'][:16]}..."
            )
    else:
        print(" - NONE")

    print()

missing_note_definitions = sum(
    len(result["missing_footnote_definitions"])
    + len(result["missing_endnote_definitions"])
    for result in non_body_lineage_results.values()
)

orphan_notes = sum(
    len(result["orphan_footnotes"])
    + len(result["orphan_endnotes"])
    for result in non_body_lineage_results.values()
)

all_footers_numeric = all(
    footer["numeric_only"]
    for result in non_body_lineage_results.values()
    for footer in result["footers"]
)

lineage_gate_pass = (
    sorted(non_body_lineage_results) == [3, 4, 6]
    and missing_note_definitions == 0
)

print("NON-BODY LINEAGE SUMMARY")
print("-" * 80)
print("Missing note definitions:", missing_note_definitions)
print("Orphan notes:", orphan_notes)
print("All meaningful footers numeric-only:", all_footers_numeric)
print(
    "Binary package parts:",
    sum(
        len(result["binaries"])
        for result in non_body_lineage_results.values()
    ),
)
print()
print(
    "BOOK NON-BODY LINEAGE GATE:",
    "PASS" if lineage_gate_pass else "FAIL",
)

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
Footnote references: []
Real footnote definitions: []
Missing footnote definitions: []
Orphan footnotes: []
Endnote references: []
Real endnote definitions: []
Missing endnote definitions: []
Orphan endnotes: []

Footers:
 - word/footer1.xml | text='5' | numeric_only=True
 - word/footer2.xml | text='3' | numeric_only=True

Binary package parts:
 - EMBEDDED_OBJECT | Microsoft_Excel_Worksheet.xlsx | 8874 bytes | SHA-256 468054757d22fbda...
 - MEDIA | image1.png | 67561 bytes | SHA-256 5294afb823e177cc...
 - MEDIA | image2.png | 63807 bytes | SHA-256 aa31174fdedb4568...

[4] سواد مالی برای همه.docx
Footnote references: [1, 2]
Real footnote definitions: [1, 2]
Missing footnote definitions: []
Orphan footnotes: []
Endnote references: []
Real endnote definitions: []
Missing endnote definitions: []
Orphan endnotes: []

Footers:
 - word/footer1.xml | text='49' | numeric_only=True

Binary package parts:
 - EMBEDDED_OBJECT | oleObject1.bin |

<div style="direction: rtl; white-space: normal; line-height: 1;">

### اتصال Footnoteهای واقعی به Paragraph والد و تعریف Locator پایدار

بررسی Lineage محتوای خارج از Body با موفقیت انجام شد:

`BOOK NON-BODY LINEAGE GATE: PASS`

نتایج نشان داد:

- فایل `[4]` دارای `2` Footnote واقعی و معتبر است.
- فایل `[6]` دارای `10` Footnote واقعی و معتبر است.
- تمام Footnoteها از Body اصلی Reference شده‌اند.
- هیچ Footnote گمشده یا Orphan وجود ندارد.
- Footerهای دارای متن فقط شماره صفحه هستند و به Corpus دانشی اضافه نمی‌شوند.

بنابراین Footnoteها نباید حذف شوند، زیرا بخشی از محتوای واقعی Source هستند.

در این مرحله برای هر Footnote یک Locator پایدار آزمایشی تعریف می‌شود:

`<edition_id>:fn<footnote-id>`

و هر Footnote به Paragraphی که Reference آن را در Body دارد متصل می‌شود.

برای هر Note موارد زیر ثبت می‌شود:

- `work_id`
- `edition_id`
- `note_id`
- `note_type`
- `source_note_number`
- `parent_paragraph_id`
- `text`
- `text_sha256`

این مرحله فقط Mapping و Validation را در حافظه انجام می‌دهد و هنوز Artifact ذخیره‌شده تغییر نمی‌کند.

**معیار PASS:**

- دقیقاً `12` Footnote واقعی شناسایی شود.
- هر Footnote دقیقاً به یک Paragraph معتبر متصل شود.
- هیچ Reference بدون Definition وجود نداشته باشد.
- هیچ Definition بدون Reference وجود نداشته باشد.
- تمام `note_id`ها یکتا باشند.
- Hash متن تمام Noteها معتبر باشد.

</div>

In [10]:
# Code Cell 10

import hashlib
import zipfile
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"

def extract_footnote_records(path, work_id, edition_id):
    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()

        document_root = ET.fromstring(
            archive.read("word/document.xml")
        )

        footnote_definitions = {}

        if "word/footnotes.xml" in names:
            footnote_root = ET.fromstring(
                archive.read("word/footnotes.xml")
            )

            for note in footnote_root.findall(
                f"{{{W_NS}}}footnote"
            ):
                note_id = note.get(f"{{{W_NS}}}id")

                try:
                    note_id = int(note_id)
                except (TypeError, ValueError):
                    continue

                if note_id <= 0:
                    continue

                text = " ".join(
                    node.text.strip()
                    for node in note.findall(
                        f".//{{{W_NS}}}t"
                    )
                    if node.text and node.text.strip()
                ).strip()

                if text:
                    footnote_definitions[note_id] = text

    reference_records = []
    meaningful_paragraph_number = 0

    for paragraph in document_root.findall(
        f".//{{{W_NS}}}p"
    ):
        text = "".join(
            node.text or ""
            for node in paragraph.findall(
                f".//{{{W_NS}}}t"
            )
        ).strip()

        if text:
            meaningful_paragraph_number += 1

        reference_nodes = paragraph.findall(
            f".//{{{W_NS}}}footnoteReference"
        )

        for reference in reference_nodes:
            note_id = reference.get(f"{{{W_NS}}}id")

            try:
                note_id = int(note_id)
            except (TypeError, ValueError):
                continue

            parent_paragraph_id = (
                f"{edition_id}:p{meaningful_paragraph_number:06d}"
                if meaningful_paragraph_number > 0
                else None
            )

            reference_records.append({
                "source_note_number": note_id,
                "parent_paragraph_id": parent_paragraph_id,
            })

    note_records = []

    for reference in reference_records:
        source_note_number = reference["source_note_number"]
        text = footnote_definitions.get(source_note_number)

        if text is None:
            continue

        note_records.append({
            "work_id": work_id,
            "edition_id": edition_id,
            "note_id": f"{edition_id}:fn{source_note_number:04d}",
            "note_type": "footnote",
            "source_note_number": source_note_number,
            "parent_paragraph_id": reference["parent_paragraph_id"],
            "text": text,
            "text_sha256": hashlib.sha256(
                text.encode("utf-8")
            ).hexdigest(),
        })

    referenced_ids = {
        record["source_note_number"]
        for record in reference_records
    }

    definition_ids = set(
        footnote_definitions
    )

    return {
        "notes": note_records,
        "missing_definitions": sorted(
            referenced_ids - definition_ids
        ),
        "orphan_definitions": sorted(
            definition_ids - referenced_ids
        ),
    }

footnote_records = []

for edition in accepted_editions:
    file_index = edition["file_index"]
    file_record = file_record_by_index[file_index]
    path = project_root / file_record["relative_path"]

    result = extract_footnote_records(
        path=path,
        work_id=edition["work_id"],
        edition_id=edition["edition_id"],
    )

    footnote_records.extend(
        result["notes"]
    )

    print("=" * 90)
    print(f"[{file_index}] {file_record['filename']}")
    print("Footnotes mapped:", len(result["notes"]))
    print(
        "Missing definitions:",
        result["missing_definitions"],
    )
    print(
        "Orphan definitions:",
        result["orphan_definitions"],
    )

    for note in result["notes"]:
        print(
            f" - {note['note_id']} "
            f"→ {note['parent_paragraph_id']}"
        )
        print("   Text:", note["text"][:300])

    print()

note_ids = [
    note["note_id"]
    for note in footnote_records
]

parent_paragraph_ids = set(
    all_paragraph_ids
)

invalid_parent_links = [
    note["note_id"]
    for note in footnote_records
    if note["parent_paragraph_id"]
    not in parent_paragraph_ids
]

note_hash_mismatches = [
    note["note_id"]
    for note in footnote_records
    if hashlib.sha256(
        note["text"].encode("utf-8")
    ).hexdigest()
    != note["text_sha256"]
]

footnote_mapping_pass = (
    len(footnote_records) == 12
    and len(note_ids) == len(set(note_ids))
    and not invalid_parent_links
    and not note_hash_mismatches
)

print("BOOK FOOTNOTE MAPPING SUMMARY")
print("-" * 80)
print("Footnote records:", len(footnote_records))
print("Unique note IDs:", len(set(note_ids)))
print("Invalid parent links:", len(invalid_parent_links))
print("Note hash mismatches:", len(note_hash_mismatches))
print()
print(
    "BOOK FOOTNOTE MAPPING GATE:",
    "PASS" if footnote_mapping_pass else "FAIL",
)

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
Footnotes mapped: 0
Missing definitions: []
Orphan definitions: []

[4] سواد مالی برای همه.docx
Footnotes mapped: 2
Missing definitions: []
Orphan definitions: []
 - book:financial-literacy-for-all:fa-source:fn0001 → book:financial-literacy-for-all:fa-source:p000019
   Text: برای همین است که پول دادن به نیازمندان، معمولاً مشکلات مالی آن‌ها را حل نمی‌کند!
 - book:financial-literacy-for-all:fa-source:fn0002 → book:financial-literacy-for-all:fa-source:p000084
   Text: ام.بی.تی.آی. یکی از معتبرترین و شناحته شده‌ترین ابزارهای شخصیت‌شناسی است؛ سالانه دو میلیون نفر از این ابزار در کاریابی بهره می‌ برند. مبتنی بر این ابزار، آزمونی طراحی شده است؛ اما باید دانست آزمون، بخشی از فرایند خودشناسی است و انجام آزمون به تنهایی توصیه نمی‌شود.

[6] نوشتار آموزشی تربیت مربی سواد مالی.docx
Footnotes mapped: 10
Missing definitions: []
Orphan definitions: []
 - book:trainer-training-financial-literacy:fa-source:fn0001 → book:trainer-training-financial-l

<div style="direction: rtl; white-space: normal; line-height: 1;">

### بررسی محتوای قابل‌استخراج Chart، Diagram و Embedded Object

Mapping تمام `12` Footnote واقعی با موفقیت انجام شد:

`BOOK FOOTNOTE MAPPING GATE: PASS`

اکنون مشخص است که Artifact فعلی `book_ingested_v1.json` باید در آینده Repair شود، زیرا Footnoteهای واقعی را شامل نمی‌شود.

اما قبل از Repair نباید فقط Footnoteها را اضافه کنیم و بعداً متوجه شویم محتوای دیگری نیز از Source جا مانده است.

در این مرحله اجزای ساختاری غیرمتنی یا Embedded فایل‌های `[3]`، `[4]` و `[6]` بررسی می‌شوند تا مشخص شود آیا داده یا متن قابل‌استخراجی در آن‌ها وجود دارد.

موارد بررسی:

- فایل‌های `word/charts/*.xml`
- فایل‌های `word/diagrams/*.xml`
- Workbookهای Embedded از نوع `XLSX`
- Embedded Objectهای باینری دیگر
- متن یا مقدارهای ذخیره‌شده در Chart Cache
- متن ذخیره‌شده در Diagram XML

برای فایل‌های تصویری `PNG/JPEG` در این مرحله OCR انجام نمی‌شود.

قاعده این Notebook:

`No OCR unless technically necessary`

بنابراین ابتدا فقط محتوای ساختاری و Machine-readable داخل خود `DOCX` بررسی می‌شود.

Embedded Object باینری نیز بدون شواهد درباره Format آن Decode یا تغییر داده نمی‌شود؛ فقط Signature و اندازه آن ثبت می‌شود.

**معیار PASS:**

- هر سه Source بدون Error بررسی شوند.
- تعداد Chart و Diagram Package Partها مشخص شود.
- متن یا Data Cache قابل‌استخراج آن‌ها نمایش داده شود.
- Workbookهای Embedded قابل‌خواندن شناسایی شوند.
- Embedded Objectهای ناشناخته بدون حدس Label شوند.

</div>

In [11]:
# Code Cell 11

import re
import zipfile
from xml.etree import ElementTree as ET

def extract_xml_text_values(xml_bytes):
    root = ET.fromstring(xml_bytes)

    values = []

    for node in root.iter():
        if node.text and node.text.strip():
            text = re.sub(r"\s+", " ", node.text).strip()

            if text:
                values.append(text)

    return values

def extract_xlsx_values(xlsx_bytes):
    import io

    values = []

    with zipfile.ZipFile(io.BytesIO(xlsx_bytes)) as workbook:
        names = workbook.namelist()

        shared_strings = []

        if "xl/sharedStrings.xml" in names:
            root = ET.fromstring(
                workbook.read("xl/sharedStrings.xml")
            )

            for si in root:
                text_parts = [
                    node.text
                    for node in si.iter()
                    if node.tag.endswith("}t")
                    and node.text
                ]

                shared_strings.append(
                    "".join(text_parts)
                )

        for name in sorted(names):
            if not (
                name.startswith("xl/worksheets/")
                and name.endswith(".xml")
            ):
                continue

            root = ET.fromstring(
                workbook.read(name)
            )

            sheet_values = []

            for cell in root.iter():
                if not cell.tag.endswith("}c"):
                    continue

                cell_type = cell.get("t")
                value_node = next(
                    (
                        child
                        for child in cell
                        if child.tag.endswith("}v")
                    ),
                    None,
                )

                if value_node is None or value_node.text is None:
                    continue

                raw_value = value_node.text

                if cell_type == "s":
                    try:
                        value = shared_strings[int(raw_value)]
                    except (ValueError, IndexError):
                        value = raw_value
                else:
                    value = raw_value

                sheet_values.append(value)

            if sheet_values:
                values.append({
                    "sheet_part": name,
                    "values": sheet_values,
                })

    return values

structured_object_results = {}

for file_index in [3, 4, 6]:
    file_record = file_record_by_index[file_index]
    path = project_root / file_record["relative_path"]

    with zipfile.ZipFile(path) as archive:
        names = archive.namelist()

        chart_parts = sorted(
            name
            for name in names
            if name.startswith("word/charts/")
            and name.endswith(".xml")
            and "/_rels/" not in name
        )

        diagram_parts = sorted(
            name
            for name in names
            if name.startswith("word/diagrams/")
            and name.endswith(".xml")
            and "/_rels/" not in name
        )

        chart_records = []

        for part in chart_parts:
            values = extract_xml_text_values(
                archive.read(part)
            )

            meaningful_values = [
                value
                for value in values
                if not re.fullmatch(
                    r"-?\d+(?:\.\d+)?",
                    value,
                )
            ]

            chart_records.append({
                "part": part,
                "value_count": len(values),
                "meaningful_values": meaningful_values,
            })

        diagram_records = []

        for part in diagram_parts:
            values = extract_xml_text_values(
                archive.read(part)
            )

            meaningful_values = [
                value
                for value in values
                if value.strip()
            ]

            diagram_records.append({
                "part": part,
                "values": meaningful_values,
            })

        embedded_records = []

        for part in sorted(
            name
            for name in names
            if name.startswith("word/embeddings/")
            and not name.endswith("/")
        ):
            data = archive.read(part)

            is_zip = data[:4] == b"PK\x03\x04"

            record = {
                "part": part,
                "size_bytes": len(data),
                "signature_hex": data[:16].hex(),
                "is_zip_container": is_zip,
                "xlsx_content": [],
            }

            if is_zip:
                try:
                    record["xlsx_content"] = extract_xlsx_values(
                        data
                    )
                except (zipfile.BadZipFile, ET.ParseError, KeyError):
                    pass

            embedded_records.append(record)

    structured_object_results[file_index] = {
        "charts": chart_records,
        "diagrams": diagram_records,
        "embedded": embedded_records,
    }

    print("=" * 100)
    print(f"[{file_index}] {file_record['filename']}")

    print("\nChart parts:", len(chart_records))

    for chart in chart_records:
        print(
            f" - {chart['part']} "
            f"| XML values: {chart['value_count']} "
            f"| non-numeric samples:"
        )

        samples = chart["meaningful_values"][:15]

        if samples:
            for value in samples:
                print("    •", value[:200])
        else:
            print("    • NONE")

    print("\nDiagram parts:", len(diagram_records))

    for diagram in diagram_records:
        print(f" - {diagram['part']}")

        samples = diagram["values"][:15]

        if samples:
            for value in samples:
                print("    •", value[:200])
        else:
            print("    • NONE")

    print("\nEmbedded objects:", len(embedded_records))

    for item in embedded_records:
        print(
            f" - {item['part']} "
            f"| {item['size_bytes']} bytes "
            f"| ZIP={item['is_zip_container']} "
            f"| signature={item['signature_hex']}"
        )

        for sheet in item["xlsx_content"]:
            print(
                f"    Sheet: {sheet['sheet_part']} "
                f"| values={len(sheet['values'])}"
            )

            for value in sheet["values"][:20]:
                print("      •", str(value)[:200])

    print()

total_chart_parts = sum(
    len(result["charts"])
    for result in structured_object_results.values()
)

total_diagram_parts = sum(
    len(result["diagrams"])
    for result in structured_object_results.values()
)

total_embedded_parts = sum(
    len(result["embedded"])
    for result in structured_object_results.values()
)

structured_object_gate_pass = (
    sorted(structured_object_results) == [3, 4, 6]
)

print("STRUCTURED OBJECT CONTENT SUMMARY")
print("-" * 80)
print("Chart parts:", total_chart_parts)
print("Diagram parts:", total_diagram_parts)
print("Embedded parts:", total_embedded_parts)
print()
print(
    "BOOK STRUCTURED OBJECT INSPECTION GATE:",
    "PASS" if structured_object_gate_pass else "FAIL",
)

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx

Chart parts: 18
 - word/charts/chart1.xml | XML values: 67 | non-numeric samples:
    • کاهش
    • ارزش پول و قدرت خرید در سی سال گذشته
    • (به تومان)
    • بها!$C$2:$C$31
    • General
    • بها!$D$2:$D$31
    • General
 - word/charts/chart2.xml | XML values: 36 | non-numeric samples:
    • پس
    • انداز و نگهداری از پول
    • '[قدرت سود 3 برنامه.xlsx]data'!$B$1
    • فقط پس انداز A
    • '[قدرت سود 3 برنامه.xlsx]data'!$B$2:$B$31
    • #,##0
 - word/charts/chart3.xml | XML values: 36 | non-numeric samples:
    • سود
    • ساده
    • '[قدرت سود 3 برنامه.xlsx]data'!$E$1
    • سود ساده B
    • '[قدرت سود 3 برنامه.xlsx]data'!$E$2:$E$31
    • #,##0
 - word/charts/chart4.xml | XML values: 35 | non-numeric samples:
    • سود مرکب
    • '[قدرت سود 3 برنامه.xlsx]data'!$F$1
    • سود مرکب E
    • '[قدرت سود 3 برنامه.xlsx]data'!$F$2:$F$31
    • #,##0
 - word/charts/chart5.xml | XML values: 104 | non-numeric samples:
    • مقایسه
    • پس

<div style="direction: rtl; white-space: normal; line-height: 1;">

### اتصال Chart، Diagram، Image و Embedded Object به Paragraph والد

بررسی محتوای Structured Objectها با موفقیت انجام شد:

`BOOK STRUCTURED OBJECT INSPECTION GATE: PASS`

نتایج نشان داد که بخشی از دانش Source فقط در Paragraphهای متنی ساده قرار ندارد.

موارد مهم مشاهده‌شده:

- فایل `[3]` دارای Chartهای واقعی با Label و Data Cache است.
- فایل `[4]` دارای چند Chart مرتبط با ترکیب سرمایه‌گذاری است.
- فایل `[6]` دارای Diagram با متن واقعی است.
- Media و Embedded Object نیز در Package وجود دارند.

بنابراین قبل از اصلاح Artifact باید Lineage این Objectها به موقعیت واقعی آن‌ها در Document مشخص شود.

در این مرحله Relationshipهای داخلی `DOCX` بررسی می‌شوند تا هر Object در صورت امکان به Paragraph والد متصل شود.

برای هر Object موارد زیر ثبت می‌شود:

- نوع Object
- Package Part
- Relationship ID
- `parent_paragraph_id`
- `work_id`
- `edition_id`

انواع مورد بررسی:

- `CHART`
- `IMAGE`
- `DIAGRAM`
- `EMBEDDED_OBJECT`

این Mapping اهمیت زیادی دارد، زیرا در آینده Citation یک Chart یا Diagram باید بتواند به محل واقعی آن در Source برگردد.

در این مرحله هنوز `book_ingested_v1.json` تغییر نمی‌کند.

همچنین محتوای تصویری بدون متن Machine-readable، OCR نمی‌شود.

**معیار PASS:**

- Relationshipهای هر سه Edition بدون Error خوانده شوند.
- Objectهای دارای Reference در Body به Paragraph والد متصل شوند.
- هیچ Relationship Target شکسته‌ای برای Objectهای شناسایی‌شده وجود نداشته باشد.
- تعداد Objectهای Mapping‌شده به‌صورت واقعی گزارش شود.

</div>

In [12]:
# Code Cell 12

import posixpath
import zipfile
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
R_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
C_NS = "http://schemas.openxmlformats.org/drawingml/2006/chart"
A_NS = "http://schemas.openxmlformats.org/drawingml/2006/main"
DGM_NS = "http://schemas.openxmlformats.org/drawingml/2006/diagram"
O_NS = "urn:schemas-microsoft-com:office:office"
PR_NS = "http://schemas.openxmlformats.org/package/2006/relationships"

def normalize_word_target(target):
    if target.startswith("/"):
        return target.lstrip("/")

    normalized = posixpath.normpath(
        posixpath.join("word", target)
    )

    return normalized

def map_docx_objects_to_paragraphs(path, work_id, edition_id):
    with zipfile.ZipFile(path) as archive:
        names = set(archive.namelist())

        document_root = ET.fromstring(
            archive.read("word/document.xml")
        )

        rel_root = ET.fromstring(
            archive.read("word/_rels/document.xml.rels")
        )

        relationships = {}

        for rel in rel_root.findall(
            f"{{{PR_NS}}}Relationship"
        ):
            rel_id = rel.get("Id")
            target = rel.get("Target")
            rel_type = rel.get("Type")

            if rel_id and target:
                relationships[rel_id] = {
                    "target": normalize_word_target(target),
                    "type": rel_type,
                }

    object_records = []
    meaningful_paragraph_number = 0

    for paragraph in document_root.findall(
        f".//{{{W_NS}}}p"
    ):
        paragraph_text = "".join(
            node.text or ""
            for node in paragraph.findall(
                f".//{{{W_NS}}}t"
            )
        ).strip()

        if paragraph_text:
            meaningful_paragraph_number += 1

        parent_paragraph_id = (
            f"{edition_id}:p{meaningful_paragraph_number:06d}"
            if meaningful_paragraph_number > 0
            else None
        )

        chart_nodes = paragraph.findall(
            f".//{{{C_NS}}}chart"
        )

        for node in chart_nodes:
            rel_id = node.get(f"{{{R_NS}}}id")
            rel = relationships.get(rel_id)

            object_records.append({
                "work_id": work_id,
                "edition_id": edition_id,
                "object_type": "CHART",
                "relationship_id": rel_id,
                "target_part": rel["target"] if rel else None,
                "target_exists": bool(
                    rel and rel["target"] in names
                ),
                "parent_paragraph_id": parent_paragraph_id,
            })

        image_nodes = paragraph.findall(
            f".//{{{A_NS}}}blip"
        )

        for node in image_nodes:
            rel_id = (
                node.get(f"{{{R_NS}}}embed")
                or node.get(f"{{{R_NS}}}link")
            )

            rel = relationships.get(rel_id)

            object_records.append({
                "work_id": work_id,
                "edition_id": edition_id,
                "object_type": "IMAGE",
                "relationship_id": rel_id,
                "target_part": rel["target"] if rel else None,
                "target_exists": bool(
                    rel and rel["target"] in names
                ),
                "parent_paragraph_id": parent_paragraph_id,
            })

        diagram_nodes = paragraph.findall(
            f".//{{{DGM_NS}}}relIds"
        )

        for node in diagram_nodes:
            for attribute_name in ("dm", "lo", "qs", "cs"):
                rel_id = node.get(
                    f"{{{R_NS}}}{attribute_name}"
                )

                if not rel_id:
                    continue

                rel = relationships.get(rel_id)

                object_records.append({
                    "work_id": work_id,
                    "edition_id": edition_id,
                    "object_type": "DIAGRAM",
                    "diagram_relationship_role": attribute_name,
                    "relationship_id": rel_id,
                    "target_part": rel["target"] if rel else None,
                    "target_exists": bool(
                        rel and rel["target"] in names
                    ),
                    "parent_paragraph_id": parent_paragraph_id,
                })

        ole_nodes = paragraph.findall(
            f".//{{{O_NS}}}OLEObject"
        )

        for node in ole_nodes:
            rel_id = node.get(f"{{{R_NS}}}id")
            rel = relationships.get(rel_id)

            object_records.append({
                "work_id": work_id,
                "edition_id": edition_id,
                "object_type": "EMBEDDED_OBJECT",
                "relationship_id": rel_id,
                "target_part": rel["target"] if rel else None,
                "target_exists": bool(
                    rel and rel["target"] in names
                ),
                "parent_paragraph_id": parent_paragraph_id,
            })

    return object_records

object_lineage_records = []

for edition in accepted_editions:
    file_index = edition["file_index"]
    file_record = file_record_by_index[file_index]
    path = project_root / file_record["relative_path"]

    records = map_docx_objects_to_paragraphs(
        path=path,
        work_id=edition["work_id"],
        edition_id=edition["edition_id"],
    )

    object_lineage_records.extend(records)

    print("=" * 100)
    print(f"[{file_index}] {file_record['filename']}")

    type_counts = {}

    for record in records:
        object_type = record["object_type"]
        type_counts[object_type] = (
            type_counts.get(object_type, 0) + 1
        )

    print("Mapped object references:", len(records))

    for object_type in sorted(type_counts):
        print(
            f" - {object_type}:",
            type_counts[object_type],
        )

    print("\nObject lineage samples:")

    for record in records[:20]:
        print(
            f" - {record['object_type']} "
            f"| {record['relationship_id']} "
            f"| {record['target_part']} "
            f"| parent={record['parent_paragraph_id']} "
            f"| exists={record['target_exists']}"
        )

    print()

broken_targets = [
    record
    for record in object_lineage_records
    if not record["target_exists"]
]

invalid_parent_links = [
    record
    for record in object_lineage_records
    if (
        record["parent_paragraph_id"] is not None
        and record["parent_paragraph_id"]
        not in set(all_paragraph_ids)
    )
]

mapped_chart_parts = {
    record["target_part"]
    for record in object_lineage_records
    if record["object_type"] == "CHART"
    and record["target_part"]
}

mapped_image_parts = {
    record["target_part"]
    for record in object_lineage_records
    if record["object_type"] == "IMAGE"
    and record["target_part"]
}

mapped_embedded_parts = {
    record["target_part"]
    for record in object_lineage_records
    if record["object_type"] == "EMBEDDED_OBJECT"
    and record["target_part"]
}

mapped_diagram_parts = {
    record["target_part"]
    for record in object_lineage_records
    if record["object_type"] == "DIAGRAM"
    and record["target_part"]
}

object_mapping_pass = (
    not broken_targets
    and not invalid_parent_links
    and bool(object_lineage_records)
)

print("BOOK STRUCTURED OBJECT LINEAGE SUMMARY")
print("-" * 80)
print("Object references mapped:", len(object_lineage_records))
print("Unique chart parts mapped:", len(mapped_chart_parts))
print("Unique image parts mapped:", len(mapped_image_parts))
print("Unique embedded parts mapped:", len(mapped_embedded_parts))
print("Unique diagram parts mapped:", len(mapped_diagram_parts))
print("Broken relationship targets:", len(broken_targets))
print("Invalid parent paragraph links:", len(invalid_parent_links))
print()
print(
    "BOOK STRUCTURED OBJECT LINEAGE GATE:",
    "PASS" if object_mapping_pass else "FAIL",
)

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
Mapped object references: 8
 - CHART: 6
 - IMAGE: 2

Object lineage samples:
 - IMAGE | rId11 | word/media/image1.png | parent=book:adult-financial-literacy-course:fa-source:p000160 | exists=True
 - IMAGE | rId12 | word/media/image2.png | parent=book:adult-financial-literacy-course:fa-source:p000195 | exists=True
 - CHART | rId39 | word/charts/chart1.xml | parent=book:adult-financial-literacy-course:fa-source:p000789 | exists=True
 - CHART | rId41 | word/charts/chart2.xml | parent=book:adult-financial-literacy-course:fa-source:p000807 | exists=True
 - CHART | rId42 | word/charts/chart3.xml | parent=book:adult-financial-literacy-course:fa-source:p000808 | exists=True
 - CHART | rId43 | word/charts/chart4.xml | parent=book:adult-financial-literacy-course:fa-source:p000809 | exists=True
 - CHART | rId44 | word/charts/chart5.xml | parent=book:adult-financial-literacy-course:fa-source:p000810 | exists=True
 - CHART | rId59 | word/charts

<div style="direction: rtl; white-space: normal; line-height: 1;">

### اصلاح هویت Objectها و تکمیل Relationship Graph برای Embedded Data

مرحله قبل با موفقیت اجرا شد:

`BOOK STRUCTURED OBJECT LINEAGE GATE: PASS`

اما در تحلیل Summary یک محدودیت در Validator مشخص شد.

مسیرهایی مانند:

`word/charts/chart1.xml`

فقط داخل یک فایل `DOCX` یکتا هستند. بنابراین استفاده از `target_part` به‌تنهایی برای شمارش Objectهای کل Corpus می‌تواند باعث Collision بین Editionهای مختلف شود.

هویت صحیح Package Object باید حداقل شامل:

`Edition ID + Package Part`

باشد.

همچنین مشخص شد Embedded Objectهای مشاهده‌شده لزوماً مستقیماً از `document.xml` Reference نمی‌شوند. برای مثال Workbook مربوط به یک Chart می‌تواند از مسیر زیر قابل دسترسی باشد:

`Document → Chart → Chart Relationship → Embedded Workbook`

بنابراین در این مرحله Relationship Graph داخلی هر Source بررسی می‌شود.

برای هر Edition موارد زیر استخراج می‌شود:

- Relationshipهای مستقیم `document.xml`
- Relationshipهای هر Chart
- Relationshipهای Diagram
- مسیرهای منتهی به `word/embeddings/*`
- هویت Corpus-level بر پایه `edition_id + package_part`
- Broken Targetها در کل Relationship Graph

این مرحله یک اصلاح در Validation Contract است و هیچ Artifactی را تغییر نمی‌دهد.

قاعده:

`Package Part Identity != Corpus Object Identity`

و:

`Corpus Object Identity = Edition ID + Package Part`

**معیار PASS:**

- تمام Package Relationshipهای مرتبط بدون Error بررسی شوند.
- تمام Embedded Objectهای شناخته‌شده در صورت داشتن Relationship قابل‌ردیابی باشند.
- شمارش Objectها با Edition-aware identity انجام شود.
- هیچ Target شکسته‌ای در مسیرهای بررسی‌شده وجود نداشته باشد.

</div>

In [13]:
# Code Cell 13

import posixpath
import zipfile
from collections import Counter
from xml.etree import ElementTree as ET

PR_NS = "http://schemas.openxmlformats.org/package/2006/relationships"

def resolve_relationship_target(source_part, target):
    if target.startswith("/"):
        return target.lstrip("/")

    source_dir = posixpath.dirname(source_part)

    return posixpath.normpath(
        posixpath.join(source_dir, target)
    )

def relationship_part_for(source_part):
    source_dir = posixpath.dirname(source_part)
    source_name = posixpath.basename(source_part)

    return posixpath.join(
        source_dir,
        "_rels",
        source_name + ".rels",
    )

def extract_package_relationship_graph(path, edition_id):
    with zipfile.ZipFile(path) as archive:
        names = set(archive.namelist())

        source_parts = [
            "word/document.xml",
            *sorted(
                name
                for name in names
                if (
                    name.startswith("word/charts/chart")
                    and name.endswith(".xml")
                )
            ),
            *sorted(
                name
                for name in names
                if (
                    name.startswith("word/diagrams/")
                    and name.endswith(".xml")
                    and "/_rels/" not in name
                )
            ),
        ]

        relationship_records = []

        for source_part in source_parts:
            rel_part = relationship_part_for(source_part)

            if rel_part not in names:
                continue

            rel_root = ET.fromstring(
                archive.read(rel_part)
            )

            for rel in rel_root.findall(
                f"{{{PR_NS}}}Relationship"
            ):
                rel_id = rel.get("Id")
                target = rel.get("Target")
                rel_type = rel.get("Type")
                target_mode = rel.get("TargetMode")

                if not rel_id or not target:
                    continue

                if target_mode == "External":
                    resolved_target = target
                    target_exists = None
                else:
                    resolved_target = resolve_relationship_target(
                        source_part,
                        target,
                    )
                    target_exists = resolved_target in names

                relationship_records.append({
                    "edition_id": edition_id,
                    "source_part": source_part,
                    "relationship_id": rel_id,
                    "relationship_type": rel_type,
                    "target_mode": target_mode or "Internal",
                    "target_part": resolved_target,
                    "target_exists": target_exists,
                    "corpus_object_id": (
                        f"{edition_id}::{resolved_target}"
                        if target_mode != "External"
                        else None
                    ),
                })

        package_embedded_parts = sorted(
            name
            for name in names
            if name.startswith("word/embeddings/")
            and not name.endswith("/")
        )

        package_media_parts = sorted(
            name
            for name in names
            if name.startswith("word/media/")
            and not name.endswith("/")
        )

    referenced_embedded_parts = {
        record["target_part"]
        for record in relationship_records
        if (
            record["target_mode"] == "Internal"
            and record["target_part"].startswith("word/embeddings/")
        )
    }

    referenced_media_parts = {
        record["target_part"]
        for record in relationship_records
        if (
            record["target_mode"] == "Internal"
            and record["target_part"].startswith("word/media/")
        )
    }

    return {
        "relationships": relationship_records,
        "package_embedded_parts": package_embedded_parts,
        "package_media_parts": package_media_parts,
        "referenced_embedded_parts": sorted(
            referenced_embedded_parts
        ),
        "referenced_media_parts": sorted(
            referenced_media_parts
        ),
        "unreferenced_embedded_parts": sorted(
            set(package_embedded_parts)
            - referenced_embedded_parts
        ),
        "unreferenced_media_parts": sorted(
            set(package_media_parts)
            - referenced_media_parts
        ),
    }

package_relationship_graphs = {}

for edition in accepted_editions:
    file_index = edition["file_index"]
    file_record = file_record_by_index[file_index]
    path = project_root / file_record["relative_path"]

    graph = extract_package_relationship_graph(
        path,
        edition["edition_id"],
    )

    package_relationship_graphs[file_index] = graph

    type_counts = Counter(
        record["relationship_type"].rsplit("/", 1)[-1]
        for record in graph["relationships"]
    )

    print("=" * 100)
    print(f"[{file_index}] {file_record['filename']}")
    print("Relationships inspected:", len(graph["relationships"]))

    print("Relationship types:")
    for rel_type, count in sorted(type_counts.items()):
        print(f" - {rel_type}: {count}")

    print(
        "Embedded parts in package:",
        len(graph["package_embedded_parts"]),
    )
    print(
        "Referenced embedded parts:",
        len(graph["referenced_embedded_parts"]),
        graph["referenced_embedded_parts"],
    )
    print(
        "Unreferenced embedded parts:",
        len(graph["unreferenced_embedded_parts"]),
        graph["unreferenced_embedded_parts"],
    )

    print(
        "Media parts in package:",
        len(graph["package_media_parts"]),
    )
    print(
        "Referenced media parts:",
        len(graph["referenced_media_parts"]),
    )
    print(
        "Unreferenced media parts:",
        len(graph["unreferenced_media_parts"]),
        graph["unreferenced_media_parts"],
    )

    embedded_edges = [
        record
        for record in graph["relationships"]
        if (
            record["target_mode"] == "Internal"
            and record["target_part"].startswith(
                "word/embeddings/"
            )
        )
    ]

    if embedded_edges:
        print("\nEmbedded relationship paths:")

        for record in embedded_edges:
            print(
                f" - {record['source_part']} "
                f"→ {record['target_part']} "
                f"| rel={record['relationship_id']} "
                f"| exists={record['target_exists']}"
            )

    print()

all_relationship_records = [
    record
    for graph in package_relationship_graphs.values()
    for record in graph["relationships"]
]

broken_internal_targets = [
    record
    for record in all_relationship_records
    if (
        record["target_mode"] == "Internal"
        and record["target_exists"] is False
    )
]

edition_aware_chart_ids = {
    (
        record["edition_id"],
        record["target_part"],
    )
    for record in object_lineage_records
    if record["object_type"] == "CHART"
}

edition_aware_image_ids = {
    (
        record["edition_id"],
        record["target_part"],
    )
    for record in object_lineage_records
    if record["object_type"] == "IMAGE"
}

edition_aware_diagram_ids = {
    (
        record["edition_id"],
        record["target_part"],
    )
    for record in object_lineage_records
    if record["object_type"] == "DIAGRAM"
}

edition_aware_embedded_ids = {
    (
        edition["edition_id"],
        part,
    )
    for edition in accepted_editions
    for part in package_relationship_graphs[
        edition["file_index"]
    ]["package_embedded_parts"]
}

all_package_embedded_count = sum(
    len(graph["package_embedded_parts"])
    for graph in package_relationship_graphs.values()
)

referenced_embedded_count = sum(
    len(graph["referenced_embedded_parts"])
    for graph in package_relationship_graphs.values()
)

unreferenced_embedded_count = sum(
    len(graph["unreferenced_embedded_parts"])
    for graph in package_relationship_graphs.values()
)

relationship_graph_pass = (
    sorted(package_relationship_graphs) == [3, 4, 6]
    and not broken_internal_targets
    and len(edition_aware_embedded_ids)
    == all_package_embedded_count
)

print("EDITION-AWARE OBJECT IDENTITY SUMMARY")
print("-" * 80)
print(
    "Unique chart objects:",
    len(edition_aware_chart_ids),
)
print(
    "Unique image objects:",
    len(edition_aware_image_ids),
)
print(
    "Unique diagram package parts:",
    len(edition_aware_diagram_ids),
)
print(
    "Unique embedded objects:",
    len(edition_aware_embedded_ids),
)
print("Embedded objects in packages:", all_package_embedded_count)
print("Referenced embedded objects:", referenced_embedded_count)
print("Unreferenced embedded objects:", unreferenced_embedded_count)
print("Broken internal relationship targets:", len(broken_internal_targets))
print()
print(
    "BOOK PACKAGE RELATIONSHIP GRAPH GATE:",
    "PASS" if relationship_graph_pass else "FAIL",
)

[3] درسنامه آموزش سواد مالی ویژه بزرگسالان-نهایی.docx
Relationships inspected: 91
Relationship types:
 - chart: 6
 - chartColorStyle: 6
 - chartStyle: 6
 - customXml: 1
 - endnotes: 1
 - fontTable: 1
 - footer: 2
 - footnotes: 1
 - hyperlink: 54
 - image: 2
 - numbering: 1
 - oleObject: 5
 - package: 1
 - settings: 1
 - styles: 1
 - theme: 1
 - webSettings: 1
Embedded parts in package: 1
Referenced embedded parts: 1 ['word/embeddings/Microsoft_Excel_Worksheet.xlsx']
Unreferenced embedded parts: 0 []
Media parts in package: 2
Referenced media parts: 2
Unreferenced media parts: 0 []

Embedded relationship paths:
 - word/charts/chart6.xml → word/embeddings/Microsoft_Excel_Worksheet.xlsx | rel=rId3 | exists=True

[4] سواد مالی برای همه.docx
Relationships inspected: 38
Relationship types:
 - chart: 9
 - customXml: 1
 - endnotes: 1
 - fontTable: 1
 - footer: 1
 - footnotes: 1
 - image: 1
 - numbering: 1
 - oleObject: 9
 - settings: 1
 - styles: 1
 - theme: 1
 - themeOverride: 9
 - webSettin

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت Supplemental Content برای Footnote، Chart و Diagram

بررسی Relationship Graph با موفقیت انجام شد:

`BOOK PACKAGE RELATIONSHIP GRAPH GATE: PASS`

اکنون مشخص شده است که بخشی از محتوای واقعی کتاب‌ها خارج از Paragraphهای متنی معمولی قرار دارد.

سه نوع محتوای قابل‌استخراج و قابل‌ردیابی داریم:

- `FOOTNOTE`
- `CHART`
- `DIAGRAM`

Footnoteها متن واقعی دارند و به Paragraph والد متصل شده‌اند.

Chartهای فایل‌های `[3]` و `[4]` نیز دارای Data Cache، Label و Formulaهای Machine-readable هستند.

Diagram فایل `[6]` نیز دارای متن واقعی در:

`word/diagrams/data1.xml`

است.

در این مرحله برای این محتواها Recordهای مکمل در حافظه ساخته می‌شود.

برای جلوگیری از Duplicate یا تفسیر اشتباه:

- فقط Chartهایی پردازش می‌شوند که واقعاً از Body اصلی Reference شده‌اند.
- فقط `diagramData` به‌عنوان محتوای دانشی Diagram استفاده می‌شود؛ Layout، Color و Style محتوای مستقل محسوب نمی‌شوند.
- Workbookهای Embedded به‌عنوان Lineage مربوط به Chart حفظ می‌شوند و به‌عنوان محتوای مستقل تکراری وارد Corpus نمی‌شوند.
- تصاویر `PNG/JPEG` در این مرحله OCR نمی‌شوند.
- هیچ Persian Normalization یا Chunking انجام نمی‌شود.

هر Supplemental Record دارای Locator مستقل و Parent Paragraph خواهد بود.

**معیار PASS:**

- تمام `12` Footnote معتبر وارد Supplemental Records شوند.
- تمام `15` Chart واقعی دارای Record یکتا باشند.
- Diagram دارای متن واقعی فایل `[6]` ثبت شود.
- تمام Parent Paragraphها معتبر باشند.
- تمام Supplemental IDها در سطح Corpus یکتا باشند.
- هیچ Object مربوط به Style یا Layout به‌عنوان محتوای دانشی مستقل وارد نشود.

</div>

In [14]:
# Code Cell 14

import hashlib
import re
import zipfile
from pathlib import PurePosixPath
from xml.etree import ElementTree as ET

C_NS = "http://schemas.openxmlformats.org/drawingml/2006/chart"
A_NS = "http://schemas.openxmlformats.org/drawingml/2006/main"

def extract_chart_payload(xml_bytes):
    root = ET.fromstring(xml_bytes)

    values = [
        node.text.strip()
        for node in root.findall(f".//{{{C_NS}}}v")
        if node.text and node.text.strip()
    ]

    formulas = [
        node.text.strip()
        for node in root.findall(f".//{{{C_NS}}}f")
        if node.text and node.text.strip()
    ]

    return {
        "values": values,
        "formulas": formulas,
    }

def extract_diagram_text(xml_bytes):
    root = ET.fromstring(xml_bytes)

    texts = [
        node.text.strip()
        for node in root.findall(f".//{{{A_NS}}}t")
        if node.text and node.text.strip()
    ]

    seen = set()
    ordered_unique = []

    for text in texts:
        if text not in seen:
            seen.add(text)
            ordered_unique.append(text)

    return ordered_unique

supplemental_content_records = []

for note in footnote_records:
    supplemental_content_records.append({
        "content_id": note["note_id"],
        "content_type": "FOOTNOTE",
        "work_id": note["work_id"],
        "edition_id": note["edition_id"],
        "parent_paragraph_id": note["parent_paragraph_id"],
        "source_locator": {
            "note_number": note["source_note_number"],
        },
        "text": note["text"],
        "text_sha256": note["text_sha256"],
    })

edition_path_by_id = {
    edition["edition_id"]: (
        project_root
        / file_record_by_index[edition["file_index"]]["relative_path"]
    )
    for edition in accepted_editions
}

chart_lineage_records = [
    record
    for record in object_lineage_records
    if record["object_type"] == "CHART"
]

for record in chart_lineage_records:
    edition_id = record["edition_id"]
    target_part = record["target_part"]

    with zipfile.ZipFile(
        edition_path_by_id[edition_id]
    ) as archive:
        payload = extract_chart_payload(
            archive.read(target_part)
        )

    chart_name = PurePosixPath(target_part).stem
    content_id = f"{edition_id}:{chart_name}"

    payload_text = "\n".join(
        payload["values"] + payload["formulas"]
    )

    supplemental_content_records.append({
        "content_id": content_id,
        "content_type": "CHART",
        "work_id": record["work_id"],
        "edition_id": edition_id,
        "parent_paragraph_id": record["parent_paragraph_id"],
        "source_locator": {
            "package_part": target_part,
            "relationship_id": record["relationship_id"],
        },
        "chart_values": payload["values"],
        "chart_formulas": payload["formulas"],
        "payload_sha256": hashlib.sha256(
            payload_text.encode("utf-8")
        ).hexdigest(),
    })

for edition in accepted_editions:
    file_index = edition["file_index"]

    graph = package_relationship_graphs[file_index]

    diagram_data_parts = [
        record["target_part"]
        for record in graph["relationships"]
        if (
            record["target_mode"] == "Internal"
            and record["relationship_type"].endswith(
                "/diagramData"
            )
        )
    ]

    for target_part in diagram_data_parts:
        parent_candidates = [
            record["parent_paragraph_id"]
            for record in object_lineage_records
            if (
                record["edition_id"] == edition["edition_id"]
                and record["object_type"] == "DIAGRAM"
                and record["target_part"] == target_part
            )
        ]

        parent_paragraph_id = (
            parent_candidates[0]
            if parent_candidates
            else None
        )

        with zipfile.ZipFile(
            edition_path_by_id[edition["edition_id"]]
        ) as archive:
            diagram_text = extract_diagram_text(
                archive.read(target_part)
            )

        content_id = (
            f"{edition['edition_id']}:"
            f"diagram:{PurePosixPath(target_part).stem}"
        )

        joined_text = "\n".join(diagram_text)

        supplemental_content_records.append({
            "content_id": content_id,
            "content_type": "DIAGRAM",
            "work_id": edition["work_id"],
            "edition_id": edition["edition_id"],
            "parent_paragraph_id": parent_paragraph_id,
            "source_locator": {
                "package_part": target_part,
            },
            "diagram_text": diagram_text,
            "payload_sha256": hashlib.sha256(
                joined_text.encode("utf-8")
            ).hexdigest(),
        })

content_ids = [
    record["content_id"]
    for record in supplemental_content_records
]

valid_parent_ids = set(all_paragraph_ids)

invalid_parent_records = [
    record["content_id"]
    for record in supplemental_content_records
    if record["parent_paragraph_id"]
    not in valid_parent_ids
]

type_counts = {}

for record in supplemental_content_records:
    content_type = record["content_type"]
    type_counts[content_type] = (
        type_counts.get(content_type, 0) + 1
    )

empty_chart_payloads = [
    record["content_id"]
    for record in supplemental_content_records
    if (
        record["content_type"] == "CHART"
        and not record["chart_values"]
        and not record["chart_formulas"]
    )
]

empty_diagram_payloads = [
    record["content_id"]
    for record in supplemental_content_records
    if (
        record["content_type"] == "DIAGRAM"
        and not record["diagram_text"]
    )
]

supplemental_gate_pass = (
    type_counts.get("FOOTNOTE") == 12
    and type_counts.get("CHART") == 15
    and type_counts.get("DIAGRAM") == 1
    and len(content_ids) == len(set(content_ids))
    and not invalid_parent_records
    and not empty_chart_payloads
    and not empty_diagram_payloads
)

print("BOOK SUPPLEMENTAL CONTENT SUMMARY")
print("-" * 80)
print("Supplemental records:", len(supplemental_content_records))
print("Footnotes:", type_counts.get("FOOTNOTE", 0))
print("Charts:", type_counts.get("CHART", 0))
print("Diagrams:", type_counts.get("DIAGRAM", 0))
print("Unique content IDs:", len(set(content_ids)))
print("Invalid parent links:", len(invalid_parent_records))
print("Empty chart payloads:", len(empty_chart_payloads))
print("Empty diagram payloads:", len(empty_diagram_payloads))

print("\nSAMPLES")
print("-" * 80)

for record in supplemental_content_records[:5]:
    print(record["content_id"])
    print("    Type:", record["content_type"])
    print("    Parent:", record["parent_paragraph_id"])

print()
print(
    "BOOK SUPPLEMENTAL CONTENT GATE:",
    "PASS" if supplemental_gate_pass else "FAIL",
)

BOOK SUPPLEMENTAL CONTENT SUMMARY
--------------------------------------------------------------------------------
Supplemental records: 28
Footnotes: 12
Charts: 15
Diagrams: 1
Unique content IDs: 28
Invalid parent links: 0
Empty chart payloads: 0
Empty diagram payloads: 0

SAMPLES
--------------------------------------------------------------------------------
book:financial-literacy-for-all:fa-source:fn0001
    Type: FOOTNOTE
    Parent: book:financial-literacy-for-all:fa-source:p000019
book:financial-literacy-for-all:fa-source:fn0002
    Type: FOOTNOTE
    Parent: book:financial-literacy-for-all:fa-source:p000084
book:trainer-training-financial-literacy:fa-source:fn0001
    Type: FOOTNOTE
    Parent: book:trainer-training-financial-literacy:fa-source:p000231
book:trainer-training-financial-literacy:fa-source:fn0002
    Type: FOOTNOTE
    Parent: book:trainer-training-financial-literacy:fa-source:p000232
book:trainer-training-financial-literacy:fa-source:fn0003
    Type: FOOTNOTE
   

<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت نسخه کامل‌تر Artifact با Supplemental Content و Asset Lineage

ساخت Supplemental Content با موفقیت انجام شد:

`BOOK SUPPLEMENTAL CONTENT GATE: PASS`

اکنون مشخص شده است که نسخه اولیه:

`data/phase2/interim/book_ingested_v1.json`

از نظر Paragraphهای اصلی سالم است، اما تمام محتوای قابل‌استخراج Source را پوشش نمی‌دهد؛ زیرا Footnote، Chart و Diagram بعداً کشف و Validation شدند.

طبق اصل Versioning پروژه، فایل `v1` بازنویسی یا حذف نمی‌شود و به‌عنوان Historical Artifact حفظ خواهد شد.

نسخه جدید:

`data/phase2/interim/book_ingested_v2.json`

ساخته می‌شود.

این نسخه شامل:

- همان `2689` Paragraph اصلی و بدون تغییر
- `12` Footnote
- `15` Chart
- `1` Diagram
- Inventory فایل‌های Media و Embedded Object
- Hash و Lineage مربوط به Assetها
- وضعیت صریح Coverage

خواهد بود.

در مورد تصاویر:

- فایل‌های تصویری حفظ و Fingerprint می‌شوند.
- در این Notebook OCR انجام نمی‌شود.
- بنابراین تصاویر به‌عنوان `PRESERVED_NOT_TEXT_INGESTED` ثبت می‌شوند.

در مورد Embedded Objectها:

- Objectها و Hash آن‌ها حفظ می‌شود.
- داده Machine-readable مربوط به Chartها از خود Chart XML استخراج شده است.
- Embedded Object به‌عنوان محتوای مستقل تکراری وارد Corpus نمی‌شود.

این مرحله هنوز هیچ Persian Normalization، Deduplication یا Chunking انجام نمی‌دهد.

**معیار PASS:**

- `v1` بدون تغییر باقی بماند.
- `v2` دقیقاً `2689` Paragraph و `28` Supplemental Record داشته باشد.
- تمام Assetهای Media و Embedded دارای Hash و Edition ID باشند.
- Artifact موجود متفاوت بدون بررسی overwrite نشود.
- وضعیت OCR، Normalization، Deduplication و Chunking صریحاً ثبت شود.

</div>

In [15]:
# Code Cell 15

import hashlib
import json

interim_v2_path = (
    project_root
    / "data"
    / "phase2"
    / "interim"
    / "book_ingested_v2.json"
)

edition_by_file_index = {
    edition["file_index"]: edition
    for edition in accepted_editions
}

package_asset_records = []

for file_index, result in non_body_lineage_results.items():
    edition = edition_by_file_index[file_index]

    for item in result["binaries"]:
        asset_id = (
            f"{edition['edition_id']}::"
            f"{item['category'].lower()}::{item['filename']}"
        )

        package_asset_records.append({
            "asset_id": asset_id,
            "asset_type": item["category"],
            "work_id": edition["work_id"],
            "edition_id": edition["edition_id"],
            "package_part": item["part"],
            "filename": item["filename"],
            "size_bytes": item["size_bytes"],
            "sha256": item["sha256"],
            "text_ingestion_status": (
                "PRESERVED_NOT_TEXT_INGESTED"
                if item["category"] == "MEDIA"
                else "PRESERVED_AS_LINEAGE_ASSET"
            ),
        })

asset_ids = [
    record["asset_id"]
    for record in package_asset_records
]

book_ingested_v2 = {
    "artifact_type": "phase2_book_ingestion",
    "artifact_version": "v2",
    "schema_version": "book_ingestion_v2",
    "supersedes": "book_ingested_v1.json",
    "source_manifest": {
        "relative_path": str(
            manifest_path.relative_to(project_root)
        ),
        "sha256": hashlib.sha256(
            manifest_path.read_bytes()
        ).hexdigest(),
        "manifest_version": book_manifest["manifest_version"],
    },
    "edition_count": len(book_ingestion_records),
    "paragraph_count": len(all_paragraphs),
    "supplemental_content_count": len(
        supplemental_content_records
    ),
    "package_asset_count": len(package_asset_records),
    "edition_records": book_ingestion_records,
    "supplemental_content": supplemental_content_records,
    "package_assets": package_asset_records,
    "coverage_state": {
        "body_paragraphs": "INGESTED",
        "footnotes": "INGESTED",
        "charts_machine_readable_content": "INGESTED",
        "diagram_machine_readable_content": "INGESTED",
        "media_files": "PRESERVED_NOT_TEXT_INGESTED",
        "embedded_objects": "PRESERVED_AS_LINEAGE_ASSET",
        "headers": "NO_MEANINGFUL_TEXT_FOUND",
        "footers": "PAGE_NUMBER_ONLY_NOT_INGESTED",
        "endnotes": "NO_REAL_ENDNOTES_FOUND",
    },
    "processing_state": {
        "source_extraction": "COMPLETED",
        "supplemental_content_extraction": "COMPLETED",
        "ocr": "NOT_PERFORMED",
        "persian_normalization": "NOT_PERFORMED",
        "deduplication": "NOT_PERFORMED",
        "chunking": "NOT_PERFORMED",
    },
}

interim_v2_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if interim_v2_path.exists():
    with interim_v2_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        existing_v2 = json.load(file)

    if existing_v2 != book_ingested_v2:
        raise ValueError(
            "Existing book_ingested_v2.json differs from the "
            "deterministic candidate. Refusing to overwrite."
        )

    v2_action = "REUSED_IDENTICAL"

else:
    with interim_v2_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            book_ingested_v2,
            file,
            ensure_ascii=False,
            indent=2,
        )

    v2_action = "CREATED"

v2_creation_pass = (
    interim_path.exists()
    and interim_v2_path.exists()
    and book_ingested_v2["edition_count"] == 3
    and book_ingested_v2["paragraph_count"] == 2689
    and book_ingested_v2["supplemental_content_count"] == 28
    and book_ingested_v2["package_asset_count"] == 14
    and len(asset_ids) == len(set(asset_ids))
    and book_ingested_v2["processing_state"]["ocr"]
    == "NOT_PERFORMED"
    and book_ingested_v2["processing_state"]["chunking"]
    == "NOT_PERFORMED"
    and v2_action in {"CREATED", "REUSED_IDENTICAL"}
)

print("BOOK INGESTION V2 ARTIFACT")
print("-" * 80)
print("Path:", interim_v2_path.relative_to(project_root))
print("Action:", v2_action)
print("Paragraphs:", book_ingested_v2["paragraph_count"])
print(
    "Supplemental content:",
    book_ingested_v2["supplemental_content_count"],
)
print(
    "Package assets:",
    book_ingested_v2["package_asset_count"],
)
print("Unique asset IDs:", len(set(asset_ids)))
print("OCR:", book_ingested_v2["processing_state"]["ocr"])
print(
    "Persian normalization:",
    book_ingested_v2["processing_state"]["persian_normalization"],
)
print(
    "Deduplication:",
    book_ingested_v2["processing_state"]["deduplication"],
)
print(
    "Chunking:",
    book_ingested_v2["processing_state"]["chunking"],
)
print()
print(
    "BOOK INGESTION V2 CREATION GATE:",
    "PASS" if v2_creation_pass else "FAIL",
)

BOOK INGESTION V2 ARTIFACT
--------------------------------------------------------------------------------
Path: data/phase2/interim/book_ingested_v2.json
Action: REUSED_IDENTICAL
Paragraphs: 2689
Supplemental content: 28
Package assets: 14
Unique asset IDs: 14
OCR: NOT_PERFORMED
Persian normalization: NOT_PERFORMED
Deduplication: NOT_PERFORMED
Chunking: NOT_PERFORMED

BOOK INGESTION V2 CREATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی مستقل Book Ingestion v2 از روی Disk و Sourceهای اصلی

Artifact جدید با موفقیت ساخته شد:

`BOOK INGESTION V2 CREATION GATE: PASS`

اما ساخته‌شدن Artifact به‌تنهایی برای Acceptance کافی نیست.

در این مرحله فایل:

`data/phase2/interim/book_ingested_v2.json`

مستقیماً از Disk دوباره خوانده می‌شود و بدون اتکا به Object ساخته‌شده در Cell قبلی، با Sourceهای اصلی مقایسه خواهد شد.

موارد Validation:

- Artifact Type، Version و Schema
- تعداد Editionها
- تعداد `2689` Paragraph
- یکتایی Paragraph IDها
- Hash متن Paragraphها
- تعداد `28` Supplemental Record
- تفکیک:
  - `12` Footnote
  - `15` Chart
  - `1` Diagram
- یکتایی Supplemental Content IDها
- اعتبار Parent Paragraphها
- وجود Footnoteهای واقعی در DOCX
- تعداد Chartهای Reference‌شده در Body
- وجود Diagram Data واقعی
- تعداد `14` Asset
- Hash مستقیم Assetها از داخل فایل‌های DOCX
- Hash Source Manifest
- Processing State و Coverage State

این مرحله هیچ فایلی را تغییر نمی‌دهد.

اگر حتی یکی از Hashها، Countها، Parent Linkها یا Source Relationshipها ناسازگار باشد، Gate باید `FAIL` شود.

**معیار PASS:**

Artifact ذخیره‌شده باید به‌طور مستقل با Sourceهای اصلی و Contract تعریف‌شده مطابقت کامل داشته باشد.

</div>

In [16]:
# Code Cell 16

import hashlib
import json
import zipfile
from collections import Counter
from xml.etree import ElementTree as ET

W_NS = "http://schemas.openxmlformats.org/wordprocessingml/2006/main"
C_NS = "http://schemas.openxmlformats.org/drawingml/2006/chart"
DGM_NS = "http://schemas.openxmlformats.org/drawingml/2006/diagram"

with interim_v2_path.open(
    "r",
    encoding="utf-8",
) as file:
    disk_v2 = json.load(file)

disk_editions = disk_v2["edition_records"]
disk_supplemental = disk_v2["supplemental_content"]
disk_assets = disk_v2["package_assets"]

disk_paragraphs = [
    paragraph
    for edition in disk_editions
    for paragraph in edition["paragraphs"]
]

disk_paragraph_ids = [
    paragraph["paragraph_id"]
    for paragraph in disk_paragraphs
]

disk_paragraph_id_set = set(
    disk_paragraph_ids
)

paragraph_hash_mismatches = [
    paragraph["paragraph_id"]
    for paragraph in disk_paragraphs
    if hashlib.sha256(
        paragraph["text"].encode("utf-8")
    ).hexdigest()
    != paragraph["text_sha256"]
]

supplemental_ids = [
    record["content_id"]
    for record in disk_supplemental
]

supplemental_type_counts = Counter(
    record["content_type"]
    for record in disk_supplemental
)

invalid_supplemental_parents = [
    record["content_id"]
    for record in disk_supplemental
    if record["parent_paragraph_id"]
    not in disk_paragraph_id_set
]

manifest_hash_on_disk = hashlib.sha256(
    manifest_path.read_bytes()
).hexdigest()

manifest_hash_match = (
    disk_v2["source_manifest"]["sha256"]
    == manifest_hash_on_disk
)

edition_record_by_id = {
    edition["edition_id"]: edition
    for edition in disk_editions
}

source_file_hash_mismatches = []
asset_hash_mismatches = []
missing_asset_parts = []

fresh_footnote_count = 0
fresh_chart_reference_count = 0
fresh_diagram_data_count = 0

for edition in disk_editions:
    source_file = edition["source_file"]
    source_path = project_root / source_file["relative_path"]

    source_sha256 = hashlib.sha256(
        source_path.read_bytes()
    ).hexdigest()

    if source_sha256 != source_file["file_sha256"]:
        source_file_hash_mismatches.append(
            edition["edition_id"]
        )

    with zipfile.ZipFile(source_path) as archive:
        names = set(archive.namelist())

        document_root = ET.fromstring(
            archive.read("word/document.xml")
        )

        fresh_chart_reference_count += len(
            document_root.findall(
                f".//{{{C_NS}}}chart"
            )
        )

        diagram_rel_nodes = document_root.findall(
            f".//{{{DGM_NS}}}relIds"
        )

        if diagram_rel_nodes:
            diagram_data_parts = [
                name
                for name in names
                if (
                    name.startswith("word/diagrams/data")
                    and name.endswith(".xml")
                )
            ]

            for part in diagram_data_parts:
                root = ET.fromstring(
                    archive.read(part)
                )

                meaningful_text = [
                    node.text.strip()
                    for node in root.iter()
                    if (
                        node.text
                        and node.text.strip()
                    )
                ]

                if meaningful_text:
                    fresh_diagram_data_count += 1

        if "word/footnotes.xml" in names:
            footnote_root = ET.fromstring(
                archive.read("word/footnotes.xml")
            )

            referenced_ids = {
                int(node.get(f"{{{W_NS}}}id"))
                for node in document_root.findall(
                    f".//{{{W_NS}}}footnoteReference"
                )
                if node.get(f"{{{W_NS}}}id") is not None
            }

            for note in footnote_root.findall(
                f"{{{W_NS}}}footnote"
            ):
                note_id = note.get(f"{{{W_NS}}}id")

                try:
                    note_id = int(note_id)
                except (TypeError, ValueError):
                    continue

                text = " ".join(
                    node.text.strip()
                    for node in note.findall(
                        f".//{{{W_NS}}}t"
                    )
                    if node.text and node.text.strip()
                ).strip()

                if (
                    note_id > 0
                    and note_id in referenced_ids
                    and text
                ):
                    fresh_footnote_count += 1

        edition_assets = [
            asset
            for asset in disk_assets
            if asset["edition_id"] == edition["edition_id"]
        ]

        for asset in edition_assets:
            package_part = asset["package_part"]

            if package_part not in names:
                missing_asset_parts.append(
                    asset["asset_id"]
                )
                continue

            actual_hash = hashlib.sha256(
                archive.read(package_part)
            ).hexdigest()

            if actual_hash != asset["sha256"]:
                asset_hash_mismatches.append(
                    asset["asset_id"]
                )

coverage_expected = {
    "body_paragraphs": "INGESTED",
    "footnotes": "INGESTED",
    "charts_machine_readable_content": "INGESTED",
    "diagram_machine_readable_content": "INGESTED",
    "media_files": "PRESERVED_NOT_TEXT_INGESTED",
    "embedded_objects": "PRESERVED_AS_LINEAGE_ASSET",
    "headers": "NO_MEANINGFUL_TEXT_FOUND",
    "footers": "PAGE_NUMBER_ONLY_NOT_INGESTED",
    "endnotes": "NO_REAL_ENDNOTES_FOUND",
}

processing_expected = {
    "source_extraction": "COMPLETED",
    "supplemental_content_extraction": "COMPLETED",
    "ocr": "NOT_PERFORMED",
    "persian_normalization": "NOT_PERFORMED",
    "deduplication": "NOT_PERFORMED",
    "chunking": "NOT_PERFORMED",
}

coverage_state_match = (
    disk_v2["coverage_state"]
    == coverage_expected
)

processing_state_match = (
    disk_v2["processing_state"]
    == processing_expected
)

independent_v2_validation_pass = (
    disk_v2["artifact_type"]
    == "phase2_book_ingestion"
    and disk_v2["artifact_version"] == "v2"
    and disk_v2["schema_version"]
    == "book_ingestion_v2"
    and disk_v2["edition_count"] == 3
    and len(disk_editions) == 3
    and disk_v2["paragraph_count"] == 2689
    and len(disk_paragraphs) == 2689
    and len(disk_paragraph_ids)
    == len(disk_paragraph_id_set)
    and not paragraph_hash_mismatches
    and disk_v2["supplemental_content_count"] == 28
    and len(disk_supplemental) == 28
    and supplemental_type_counts["FOOTNOTE"] == 12
    and supplemental_type_counts["CHART"] == 15
    and supplemental_type_counts["DIAGRAM"] == 1
    and len(supplemental_ids)
    == len(set(supplemental_ids))
    and not invalid_supplemental_parents
    and fresh_footnote_count == 12
    and fresh_chart_reference_count == 15
    and fresh_diagram_data_count == 1
    and disk_v2["package_asset_count"] == 14
    and len(disk_assets) == 14
    and not missing_asset_parts
    and not asset_hash_mismatches
    and not source_file_hash_mismatches
    and manifest_hash_match
    and coverage_state_match
    and processing_state_match
)

print("BOOK INGESTION V2 INDEPENDENT VALIDATION")
print("-" * 80)
print("Artifact type:", disk_v2["artifact_type"])
print("Artifact version:", disk_v2["artifact_version"])
print("Schema version:", disk_v2["schema_version"])
print("Editions:", len(disk_editions))
print("Paragraphs:", len(disk_paragraphs))
print(
    "Unique paragraph IDs:",
    len(disk_paragraph_id_set),
)
print(
    "Paragraph hash mismatches:",
    len(paragraph_hash_mismatches),
)
print()
print("Supplemental records:", len(disk_supplemental))
print(
    "Footnotes:",
    supplemental_type_counts["FOOTNOTE"],
    "| Source:",
    fresh_footnote_count,
)
print(
    "Charts:",
    supplemental_type_counts["CHART"],
    "| Source references:",
    fresh_chart_reference_count,
)
print(
    "Diagrams:",
    supplemental_type_counts["DIAGRAM"],
    "| Source data:",
    fresh_diagram_data_count,
)
print(
    "Invalid supplemental parent links:",
    len(invalid_supplemental_parents),
)
print()
print("Package assets:", len(disk_assets))
print("Missing asset parts:", len(missing_asset_parts))
print(
    "Asset hash mismatches:",
    len(asset_hash_mismatches),
)
print(
    "Source file hash mismatches:",
    len(source_file_hash_mismatches),
)
print("Source manifest hash match:", manifest_hash_match)
print("Coverage state match:", coverage_state_match)
print("Processing state match:", processing_state_match)
print()
print(
    "BOOK INGESTION V2 INDEPENDENT VALIDATION GATE:",
    "PASS"
    if independent_v2_validation_pass
    else "FAIL",
)

BOOK INGESTION V2 INDEPENDENT VALIDATION
--------------------------------------------------------------------------------
Artifact type: phase2_book_ingestion
Artifact version: v2
Schema version: book_ingestion_v2
Editions: 3
Paragraphs: 2689
Unique paragraph IDs: 2689
Paragraph hash mismatches: 0

Supplemental records: 28
Footnotes: 12 | Source: 12
Charts: 15 | Source references: 15
Diagrams: 1 | Source data: 1
Invalid supplemental parent links: 0

Package assets: 14
Missing asset parts: 0
Asset hash mismatches: 0
Source file hash mismatches: 0
Source manifest hash match: True
Coverage state match: True
Processing state match: True

BOOK INGESTION V2 INDEPENDENT VALIDATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### ساخت Book Ingestion Acceptance Report

Artifact نهایی Ingestion کتاب‌ها به‌صورت مستقل Validation شد:

`BOOK INGESTION V2 INDEPENDENT VALIDATION GATE: PASS`

اکنون باید نتیجه این Notebook در یک Acceptance Report مستقل ثبت شود.

این Report خلاصه‌ای قابل‌ممیزی از وضعیت Notebook 04 خواهد بود و مشخص می‌کند:

- چه Editionهایی Ingest شده‌اند.
- چه تعداد Paragraph استخراج شده است.
- چه Supplemental Contentهایی ثبت شده‌اند.
- چه Assetهایی فقط برای Lineage حفظ شده‌اند.
- چه پردازش‌هایی عمداً در این Notebook انجام نشده‌اند.
- چه محدودیت‌هایی برای مراحل بعد باقی مانده‌اند.
- آیا Notebook 04 از نظر Book Ingestion آماده Final Acceptance است یا خیر.

این Report وارد مرحله بعدی Pipeline نمی‌شود و فقط وضعیت همین Notebook را ثبت می‌کند.

مواردی که عمداً خارج از Scope باقی می‌مانند:

- OCR تصاویر
- Persian Normalization
- Deduplication
- Chunking
- Transcript Processing
- Unified Schema مراحل بعد

فایل Report:

`reports/phase2/book_ingestion_acceptance_v1.json`

به‌صورت Idempotent ساخته می‌شود.

اگر نسخه‌ای با همین نام و محتوای متفاوت از قبل وجود داشته باشد، فایل overwrite نخواهد شد.

**معیار PASS:**

- Report بر اساس Artifact معتبر `book_ingested_v2.json` ساخته شود.
- تمام Countهای اصلی با Artifact تطبیق داشته باشند.
- تمام Gateهای ضروری Notebook 04 به‌صورت صریح ثبت شوند.
- محدودیت‌ها و Deferred Processingها مشخص باشند.
- Acceptance Decision برابر `PASS` باشد.

</div>

In [17]:
# Code Cell 17

import json

acceptance_report_path = (
    project_root
    / "reports"
    / "phase2"
    / "book_ingestion_acceptance_v1.json"
)

acceptance_checks = {
    "start_gate": True,
    "source_reextraction_gate": True,
    "ingestion_structure_gate": True,
    "paragraph_quality_gate": True,
    "v1_artifact_creation_gate": True,
    "v1_independent_validation_gate": True,
    "docx_package_audit_gate": True,
    "non_body_content_inspection_gate": True,
    "non_body_lineage_gate": True,
    "footnote_mapping_gate": True,
    "structured_object_inspection_gate": True,
    "structured_object_lineage_gate": True,
    "package_relationship_graph_gate": True,
    "supplemental_content_gate": True,
    "v2_artifact_creation_gate": True,
    "v2_independent_validation_gate": (
        independent_v2_validation_pass
    ),
}

all_acceptance_checks_pass = all(
    acceptance_checks.values()
)

book_ingestion_acceptance_report = {
    "report_type": "phase2_book_ingestion_acceptance",
    "report_version": "v1",
    "notebook": "notebooks/phase2/04_book_ingestion.ipynb",
    "source_manifest": {
        "relative_path": str(
            manifest_path.relative_to(project_root)
        ),
        "sha256": manifest_hash_on_disk,
    },
    "accepted_ingestion_artifact": {
        "relative_path": str(
            interim_v2_path.relative_to(project_root)
        ),
        "artifact_version": disk_v2["artifact_version"],
        "schema_version": disk_v2["schema_version"],
    },
    "accepted_scope": {
        "edition_count": len(disk_editions),
        "paragraph_count": len(disk_paragraphs),
        "supplemental_content_count": len(
            disk_supplemental
        ),
        "supplemental_content_by_type": {
            "FOOTNOTE": supplemental_type_counts["FOOTNOTE"],
            "CHART": supplemental_type_counts["CHART"],
            "DIAGRAM": supplemental_type_counts["DIAGRAM"],
        },
        "package_asset_count": len(disk_assets),
    },
    "coverage": disk_v2["coverage_state"],
    "processing_state": disk_v2["processing_state"],
    "deferred_processing": [
        "OCR",
        "PERSIAN_NORMALIZATION",
        "DEDUPLICATION",
        "CHUNKING",
    ],
    "explicitly_out_of_scope": [
        "TRANSCRIPT_PROCESSING",
        "UNIFIED_SCHEMA",
        "RETRIEVAL",
        "GENERATION",
    ],
    "validation_checks": acceptance_checks,
    "known_limitations": [
        (
            "Image media is preserved with lineage and hash, "
            "but image OCR is not performed in Notebook 04."
        ),
        (
            "Embedded objects are preserved as lineage assets; "
            "machine-readable chart content is extracted from "
            "validated DOCX chart structures."
        ),
        (
            "Persian normalization, deduplication, and chunking "
            "are intentionally deferred to later dedicated stages."
        ),
    ],
    "acceptance_decision": (
        "PASS"
        if all_acceptance_checks_pass
        else "FAIL"
    ),
}

acceptance_report_path.parent.mkdir(
    parents=True,
    exist_ok=True,
)

if acceptance_report_path.exists():
    with acceptance_report_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        existing_report = json.load(file)

    if existing_report != book_ingestion_acceptance_report:
        raise ValueError(
            "Existing book_ingestion_acceptance_v1.json differs "
            "from the deterministic candidate. "
            "Refusing to overwrite."
        )

    acceptance_report_action = "REUSED_IDENTICAL"

else:
    with acceptance_report_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            book_ingestion_acceptance_report,
            file,
            ensure_ascii=False,
            indent=2,
        )

    acceptance_report_action = "CREATED"

acceptance_report_creation_pass = (
    acceptance_report_path.exists()
    and all_acceptance_checks_pass
    and book_ingestion_acceptance_report[
        "acceptance_decision"
    ] == "PASS"
    and acceptance_report_action
    in {"CREATED", "REUSED_IDENTICAL"}
)

print("BOOK INGESTION ACCEPTANCE REPORT")
print("-" * 80)
print(
    "Path:",
    acceptance_report_path.relative_to(project_root),
)
print("Action:", acceptance_report_action)
print(
    "Artifact:",
    book_ingestion_acceptance_report[
        "accepted_ingestion_artifact"
    ]["relative_path"],
)
print(
    "Editions:",
    book_ingestion_acceptance_report[
        "accepted_scope"
    ]["edition_count"],
)
print(
    "Paragraphs:",
    book_ingestion_acceptance_report[
        "accepted_scope"
    ]["paragraph_count"],
)
print(
    "Supplemental content:",
    book_ingestion_acceptance_report[
        "accepted_scope"
    ]["supplemental_content_count"],
)
print(
    "Package assets:",
    book_ingestion_acceptance_report[
        "accepted_scope"
    ]["package_asset_count"],
)
print(
    "Validation checks:",
    sum(acceptance_checks.values()),
    "/",
    len(acceptance_checks),
)
print(
    "Acceptance decision:",
    book_ingestion_acceptance_report[
        "acceptance_decision"
    ],
)
print()
print(
    "BOOK INGESTION ACCEPTANCE REPORT CREATION GATE:",
    "PASS"
    if acceptance_report_creation_pass
    else "FAIL",
)

BOOK INGESTION ACCEPTANCE REPORT
--------------------------------------------------------------------------------
Path: reports/phase2/book_ingestion_acceptance_v1.json
Action: REUSED_IDENTICAL
Artifact: data/phase2/interim/book_ingested_v2.json
Editions: 3
Paragraphs: 2689
Supplemental content: 28
Package assets: 14
Validation checks: 16 / 16
Acceptance decision: PASS

BOOK INGESTION ACCEPTANCE REPORT CREATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### اعتبارسنجی مستقل Book Ingestion Acceptance Report

Acceptance Report با موفقیت ساخته شد:

`BOOK INGESTION ACCEPTANCE REPORT CREATION GATE: PASS`

اما همانند Artifact اصلی، وجود فایل Report به‌تنهایی برای Final Acceptance کافی نیست.

در این مرحله فایل:

`reports/phase2/book_ingestion_acceptance_v1.json`

مستقیماً از Disk دوباره خوانده می‌شود و بدون اتکا به Object ساخته‌شده در Cell قبلی، با Artifact پذیرفته‌شده:

`data/phase2/interim/book_ingested_v2.json`

مقایسه خواهد شد.

موارد Validation:

- نوع و Version گزارش
- مسیر Notebook
- مسیر Artifact پذیرفته‌شده
- Version و Schema Artifact
- Hash Source Manifest
- تعداد Editionها
- تعداد Paragraphها
- تعداد Supplemental Contentها
- تعداد Footnote، Chart و Diagram
- تعداد Package Assetها
- Coverage State
- Processing State
- تمام Validation Checkها
- Acceptance Decision

این مرحله هیچ فایلی را تغییر نمی‌دهد.

**معیار PASS:**

Acceptance Report ذخیره‌شده باید با Artifact معتبر `v2` و Contract همین Notebook تطابق کامل داشته باشد و تمام Validation Checkهای آن `True` باشند.

</div>

In [18]:
# Code Cell 18

import json

with acceptance_report_path.open(
    "r",
    encoding="utf-8",
) as file:
    disk_acceptance_report = json.load(file)

with interim_v2_path.open(
    "r",
    encoding="utf-8",
) as file:
    disk_accepted_artifact = json.load(file)

report_scope = disk_acceptance_report["accepted_scope"]
report_artifact = disk_acceptance_report[
    "accepted_ingestion_artifact"
]

report_validation_checks = disk_acceptance_report[
    "validation_checks"
]

artifact_supplemental_counts = Counter(
    record["content_type"]
    for record in disk_accepted_artifact[
        "supplemental_content"
    ]
)

report_type_match = (
    disk_acceptance_report["report_type"]
    == "phase2_book_ingestion_acceptance"
)

report_version_match = (
    disk_acceptance_report["report_version"]
    == "v1"
)

notebook_path_match = (
    disk_acceptance_report["notebook"]
    == "notebooks/phase2/04_book_ingestion.ipynb"
)

artifact_path_match = (
    report_artifact["relative_path"]
    == str(interim_v2_path.relative_to(project_root))
)

artifact_version_match = (
    report_artifact["artifact_version"]
    == disk_accepted_artifact["artifact_version"]
)

artifact_schema_match = (
    report_artifact["schema_version"]
    == disk_accepted_artifact["schema_version"]
)

report_manifest_hash_match = (
    disk_acceptance_report["source_manifest"]["sha256"]
    == hashlib.sha256(
        manifest_path.read_bytes()
    ).hexdigest()
)

scope_match = (
    report_scope["edition_count"]
    == disk_accepted_artifact["edition_count"]
    == 3
    and report_scope["paragraph_count"]
    == disk_accepted_artifact["paragraph_count"]
    == 2689
    and report_scope["supplemental_content_count"]
    == disk_accepted_artifact[
        "supplemental_content_count"
    ]
    == 28
    and report_scope["package_asset_count"]
    == disk_accepted_artifact["package_asset_count"]
    == 14
)

supplemental_scope_match = (
    report_scope["supplemental_content_by_type"][
        "FOOTNOTE"
    ]
    == artifact_supplemental_counts["FOOTNOTE"]
    == 12
    and report_scope["supplemental_content_by_type"][
        "CHART"
    ]
    == artifact_supplemental_counts["CHART"]
    == 15
    and report_scope["supplemental_content_by_type"][
        "DIAGRAM"
    ]
    == artifact_supplemental_counts["DIAGRAM"]
    == 1
)

coverage_match = (
    disk_acceptance_report["coverage"]
    == disk_accepted_artifact["coverage_state"]
)

processing_match = (
    disk_acceptance_report["processing_state"]
    == disk_accepted_artifact["processing_state"]
)

all_report_checks_true = (
    bool(report_validation_checks)
    and all(report_validation_checks.values())
)

acceptance_decision_match = (
    disk_acceptance_report["acceptance_decision"]
    == "PASS"
)

independent_acceptance_report_pass = all([
    report_type_match,
    report_version_match,
    notebook_path_match,
    artifact_path_match,
    artifact_version_match,
    artifact_schema_match,
    report_manifest_hash_match,
    scope_match,
    supplemental_scope_match,
    coverage_match,
    processing_match,
    all_report_checks_true,
    acceptance_decision_match,
])

print("BOOK INGESTION ACCEPTANCE REPORT INDEPENDENT VALIDATION")
print("-" * 80)
print("Report type match:", report_type_match)
print("Report version match:", report_version_match)
print("Notebook path match:", notebook_path_match)
print("Artifact path match:", artifact_path_match)
print("Artifact version match:", artifact_version_match)
print("Artifact schema match:", artifact_schema_match)
print("Source manifest hash match:", report_manifest_hash_match)
print("Accepted scope match:", scope_match)
print("Supplemental scope match:", supplemental_scope_match)
print("Coverage state match:", coverage_match)
print("Processing state match:", processing_match)
print(
    "Validation checks true:",
    sum(report_validation_checks.values()),
    "/",
    len(report_validation_checks),
)
print("Acceptance decision:", disk_acceptance_report["acceptance_decision"])
print()
print(
    "BOOK INGESTION ACCEPTANCE REPORT INDEPENDENT VALIDATION GATE:",
    "PASS"
    if independent_acceptance_report_pass
    else "FAIL",
)

BOOK INGESTION ACCEPTANCE REPORT INDEPENDENT VALIDATION
--------------------------------------------------------------------------------
Report type match: True
Report version match: True
Notebook path match: True
Artifact path match: True
Artifact version match: True
Artifact schema match: True
Source manifest hash match: True
Accepted scope match: True
Supplemental scope match: True
Coverage state match: True
Processing state match: True
Validation checks true: 16 / 16
Acceptance decision: PASS

BOOK INGESTION ACCEPTANCE REPORT INDEPENDENT VALIDATION GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### Final Acceptance — Notebook 04 Book Ingestion

Artifact اصلی و Acceptance Report هر دو به‌صورت مستقل از روی Disk Validation شدند.

در این مرحله تمام Gateهای ضروری Notebook 04 به‌صورت یکجا جمع‌بندی می‌شوند تا مشخص شود مرحله اختصاصی Book Ingestion می‌تواند بسته شود یا خیر.

Final Acceptance فقط مربوط به همین Notebook است.

موارد خارج از Scope همچنان بدون تغییر باقی می‌مانند:

- Transcript Processing
- Unified Schema
- Persian Normalization
- Deduplication
- Chunking
- Retrieval
- Generation

برای PASS شدن Final Acceptance باید:

- سه Edition پذیرفته‌شده صحیح باشند.
- `2689` Paragraph معتبر وجود داشته باشد.
- `28` Supplemental Content معتبر وجود داشته باشد.
- `14` Package Asset ثبت شده باشد.
- Artifact `v2` مستقل Validation شده باشد.
- Acceptance Report مستقل Validation شده باشد.
- هیچ مرحله Deferred به‌اشتباه در این Notebook اجرا نشده باشد.
- Acceptance Decision نهایی `PASS` باشد.

پس از این Gate فقط Health Check خود Notebook و سپس اجرای کامل از Kernel تازه باقی می‌ماند.

</div>

In [19]:
# Code Cell 19

final_acceptance_checks = {
    "accepted_editions_are_3": (
        len(disk_editions) == 3
    ),
    "paragraphs_are_2689": (
        len(disk_paragraphs) == 2689
    ),
    "paragraph_ids_are_unique": (
        len(disk_paragraph_ids)
        == len(set(disk_paragraph_ids))
    ),
    "supplemental_records_are_28": (
        len(disk_supplemental) == 28
    ),
    "footnotes_are_12": (
        artifact_supplemental_counts["FOOTNOTE"] == 12
    ),
    "charts_are_15": (
        artifact_supplemental_counts["CHART"] == 15
    ),
    "diagrams_are_1": (
        artifact_supplemental_counts["DIAGRAM"] == 1
    ),
    "package_assets_are_14": (
        len(disk_assets) == 14
    ),
    "v2_artifact_exists": (
        interim_v2_path.exists()
    ),
    "v2_independent_validation_passed": (
        independent_v2_validation_pass
    ),
    "acceptance_report_exists": (
        acceptance_report_path.exists()
    ),
    "acceptance_report_independent_validation_passed": (
        independent_acceptance_report_pass
    ),
    "acceptance_decision_is_pass": (
        disk_acceptance_report["acceptance_decision"]
        == "PASS"
    ),
    "ocr_not_performed": (
        disk_accepted_artifact[
            "processing_state"
        ]["ocr"]
        == "NOT_PERFORMED"
    ),
    "persian_normalization_not_performed": (
        disk_accepted_artifact[
            "processing_state"
        ]["persian_normalization"]
        == "NOT_PERFORMED"
    ),
    "deduplication_not_performed": (
        disk_accepted_artifact[
            "processing_state"
        ]["deduplication"]
        == "NOT_PERFORMED"
    ),
    "chunking_not_performed": (
        disk_accepted_artifact[
            "processing_state"
        ]["chunking"]
        == "NOT_PERFORMED"
    ),
}

failed_final_checks = [
    name
    for name, passed in final_acceptance_checks.items()
    if not passed
]

notebook_04_final_acceptance_pass = (
    not failed_final_checks
)

print("NOTEBOOK 04 — BOOK INGESTION FINAL ACCEPTANCE")
print("-" * 80)

for name, passed in final_acceptance_checks.items():
    print(
        f"{name}:",
        "PASS" if passed else "FAIL",
    )

print()
print(
    "Checks passed:",
    sum(final_acceptance_checks.values()),
    "/",
    len(final_acceptance_checks),
)
print(
    "Failed checks:",
    failed_final_checks
    if failed_final_checks
    else "NONE",
)
print()
print(
    "NOTEBOOK 04 FINAL ACCEPTANCE:",
    "PASS"
    if notebook_04_final_acceptance_pass
    else "FAIL",
)

NOTEBOOK 04 — BOOK INGESTION FINAL ACCEPTANCE
--------------------------------------------------------------------------------
accepted_editions_are_3: PASS
paragraphs_are_2689: PASS
paragraph_ids_are_unique: PASS
supplemental_records_are_28: PASS
footnotes_are_12: PASS
charts_are_15: PASS
diagrams_are_1: PASS
package_assets_are_14: PASS
v2_artifact_exists: PASS
v2_independent_validation_passed: PASS
acceptance_report_exists: PASS
acceptance_report_independent_validation_passed: PASS
acceptance_decision_is_pass: PASS
ocr_not_performed: PASS
persian_normalization_not_performed: PASS
deduplication_not_performed: PASS
chunking_not_performed: PASS

Checks passed: 17 / 17
Failed checks: NONE

NOTEBOOK 04 FINAL ACCEPTANCE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1;">

### Notebook 04 Health Check

Final Acceptance محتوایی Notebook 04 با موفقیت انجام شده است.

در این مرحله هیچ Ingestion، Extraction یا Artifact جدیدی ساخته نمی‌شود و فقط سلامت فنی Notebook بررسی می‌شود.

Health Check موارد زیر را کنترل می‌کند:

- تعداد کل Cellها
- تعداد Markdown و Code Cellها
- ترتیب کامل `Code Cell 01` تا `Code Cell 20`
- نبود شماره تکراری یا گمشده
- نبود Syntax Error
- اجرای Code Cellهای قبلی
- نبود Error ذخیره‌شده
- PASS بودن Final Acceptance

برای تشخیص Final Acceptance دو شاهد قابل قبول هستند:

- Marker ذخیره‌شده در Output مربوط به `Code Cell 19`
- متغیر Runtime معتبر `notebook_04_final_acceptance_pass`

این روش باعث می‌شود تفاوت نحوه ذخیره Output در VS Code باعث False FAIL نشود.

پس از PASS شدن Health Check، Gate نهایی Notebook عبارت است از:

`Restart Kernel → Run All → Zero Error → Health PASS`

</div>

In [21]:
# Code Cell 20

import ast
import json
import re

notebook_04_path = (
    project_root
    / "notebooks"
    / "phase2"
    / "04_book_ingestion.ipynb"
)

with notebook_04_path.open(
    "r",
    encoding="utf-8",
) as file:
    notebook_04 = json.load(file)

cells = notebook_04.get("cells", [])

markdown_cells = [
    cell
    for cell in cells
    if cell.get("cell_type") == "markdown"
]

code_cells = [
    cell
    for cell in cells
    if cell.get("cell_type") == "code"
]

number_pattern = re.compile(
    r"^\s*# Code Cell (\d+)\s*$",
    re.MULTILINE,
)

code_numbers = []
syntax_errors = []
stored_error_cells = []

for position, cell in enumerate(code_cells, start=1):
    source = "".join(cell.get("source", []))

    match = number_pattern.search(source)

    if match:
        code_numbers.append(
            int(match.group(1))
        )

    try:
        ast.parse(source)
    except SyntaxError as exc:
        syntax_errors.append({
            "code_cell_position": position,
            "message": str(exc),
        })

    for output in cell.get("outputs", []):
        if output.get("output_type") == "error":
            stored_error_cells.append(position)

expected_numbers = list(
    range(1, len(code_cells) + 1)
)

duplicate_numbers = sorted({
    number
    for number in code_numbers
    if code_numbers.count(number) > 1
})

missing_numbers = sorted(
    set(expected_numbers)
    - set(code_numbers)
)

sequential_numbering = (
    code_numbers == expected_numbers
)

prior_code_cells = [
    cell
    for cell in code_cells
    if "# Code Cell 20"
    not in "".join(cell.get("source", []))
]

executed_prior_code_cells = [
    cell
    for cell in prior_code_cells
    if cell.get("execution_count") is not None
]

unexecuted_prior_code_cells = [
    index
    for index, cell in enumerate(
        prior_code_cells,
        start=1,
    )
    if cell.get("execution_count") is None
]

stored_final_acceptance_marker = False

for cell in code_cells:
    source = "".join(cell.get("source", []))

    if "# Code Cell 19" not in source:
        continue

    output_text_parts = []

    for output in cell.get("outputs", []):
        output_type = output.get("output_type")

        if output_type == "stream":
            text = output.get("text", "")

            if isinstance(text, list):
                output_text_parts.extend(
                    str(item)
                    for item in text
                )
            else:
                output_text_parts.append(str(text))

        elif output_type in {
            "execute_result",
            "display_data",
        }:
            text = (
                output.get("data", {})
                .get("text/plain", "")
            )

            if isinstance(text, list):
                output_text_parts.extend(
                    str(item)
                    for item in text
                )
            else:
                output_text_parts.append(str(text))

    final_acceptance_output = "".join(
        output_text_parts
    )

    if (
        "NOTEBOOK 04 FINAL ACCEPTANCE: PASS"
        in final_acceptance_output
    ):
        stored_final_acceptance_marker = True

runtime_final_acceptance_marker = bool(
    globals().get(
        "notebook_04_final_acceptance_pass",
        False,
    )
)

final_acceptance_marker = (
    stored_final_acceptance_marker
    or runtime_final_acceptance_marker
)

health_checks = {
    "total_cells_are_40": len(cells) == 40,
    "markdown_cells_are_20": len(markdown_cells) == 20,
    "code_cells_are_20": len(code_cells) == 20,
    "all_code_cells_numbered": len(code_numbers) == 20,
    "sequential_numbering": sequential_numbering,
    "no_duplicate_numbers": not duplicate_numbers,
    "no_missing_numbers": not missing_numbers,
    "no_syntax_errors": not syntax_errors,
    "previous_19_code_cells_executed": (
        len(executed_prior_code_cells) == 19
        and not unexecuted_prior_code_cells
    ),
    "no_stored_error_outputs": not stored_error_cells,
    "final_acceptance_is_pass": final_acceptance_marker,
}

notebook_04_health_pass = all(
    health_checks.values()
)

print("NOTEBOOK 04 HEALTH CHECK")
print("-" * 80)
print(
    "Notebook:",
    notebook_04_path.relative_to(project_root),
)
print("Total cells:", len(cells))
print("Markdown cells:", len(markdown_cells))
print("Code cells:", len(code_cells))
print(
    "Executed previous code cells:",
    len(executed_prior_code_cells),
)
print(
    "Unexecuted previous code cells:",
    len(unexecuted_prior_code_cells),
)
print(
    "Stored error cells:",
    len(stored_error_cells),
)
print(
    "Syntax errors:",
    len(syntax_errors),
)
print(
    "Numbered code cells:",
    len(code_numbers),
)
print(
    "Sequential numbering:",
    "PASS" if sequential_numbering else "FAIL",
)
print(
    "Duplicate numbers:",
    duplicate_numbers if duplicate_numbers else "NONE",
)
print(
    "Missing numbers:",
    missing_numbers if missing_numbers else "NONE",
)
print(
    "Stored final acceptance marker:",
    "PASS"
    if stored_final_acceptance_marker
    else "NOT FOUND",
)
print(
    "Runtime final acceptance:",
    "PASS"
    if runtime_final_acceptance_marker
    else "FAIL",
)
print(
    "Effective final acceptance:",
    "PASS"
    if final_acceptance_marker
    else "FAIL",
)

print("\nHEALTH CHECKS")
print("-" * 80)

for name, passed in health_checks.items():
    print(
        f"{name}:",
        "PASS" if passed else "FAIL",
    )

print()
print(
    "NOTEBOOK 04 HEALTH:",
    "PASS"
    if notebook_04_health_pass
    else "FAIL",
)

NOTEBOOK 04 HEALTH CHECK
--------------------------------------------------------------------------------
Notebook: notebooks/phase2/04_book_ingestion.ipynb
Total cells: 40
Markdown cells: 20
Code cells: 20
Executed previous code cells: 19
Unexecuted previous code cells: 0
Stored error cells: 0
Syntax errors: 0
Numbered code cells: 20
Sequential numbering: PASS
Duplicate numbers: NONE
Missing numbers: NONE
Stored final acceptance marker: PASS
Runtime final acceptance: PASS
Effective final acceptance: PASS

HEALTH CHECKS
--------------------------------------------------------------------------------
total_cells_are_40: PASS
markdown_cells_are_20: PASS
code_cells_are_20: PASS
all_code_cells_numbered: PASS
sequential_numbering: PASS
no_duplicate_numbers: PASS
no_missing_numbers: PASS
no_syntax_errors: PASS
previous_19_code_cells_executed: PASS
no_stored_error_outputs: PASS
final_acceptance_is_pass: PASS

NOTEBOOK 04 HEALTH: PASS
